# Phase 2 – Data Cleaning and Analysis

## Purpose

This notebook builds on the exploratory analysis completed during Phase 1.

The goal of Phase 2 is to systematically clean and prepare the CMS Inpatient Rehabilitation Facility (IRF) provider level dataset for further analysis and provider level anomaly detection.

Initial data preparation will focus on:

- Reviewing dataset structure and variable types
- Identifying missing, suppressed, and zero values
- Evaluating patterns of missingness and CMS suppression
- Identifying duplicate or potentially invalid records
- Assessing individual variables for analytical usefulness
- Documenting decisions to retain, transform, investigate, or exclude variables
- Preserving potential outliers for further investigation rather than automatically removing them

Following data preparation, the cleaned dataset will be used to investigate Medicare payment variation and develop methods for identifying unusual provider level patterns.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [2]:
import os

os.listdir()

['CMS_IRF_2024_Cleaned_Engineered.csv',
 'Phase 2 Data Cleaning 09_27_26.ipynb',
 'RY26_PACPUF_IRF_2024_main_S.csv',
 'CMS_IRF_2024_Variable_Audit.csv',
 'Medicare Post-Acute Care Utilization - Inpatient Rehabilitation Facility by Geography and Provider.zip',
 'Phase 1 final edit 091726.ipynb',
 'Phase 1 exploratory analysis Sep 10 .ipynb',
 '.ipynb_checkpoints',
 'data']

In [3]:
df_raw = pd.read_csv("RY26_PACPUF_IRF_2024_main_S.csv")

print("Rows:", df_raw.shape[0])
print("Columns:", df_raw.shape[1])

df_raw.head()

Rows: 1218
Columns: 82


,YEAR,YEAR_TYPE,SMRY_CTGRY,SRVC_CTGRY,PRVDR_ID,PRVDR_NAME,PRVDR_CITY,STATE,PRVDR_ZIP,BENE_DSTNCT_CNT,TOT_EPSD_STAY_CNT,TOT_SRVC_DAYS,TOT_CHRG_AMT,TOT_ALOWD_AMT,TOT_MDCR_PYMT_AMT,TOT_MDCR_STDZD_PYMT_AMT,TOT_OUTLIER_PYMT_AMT,BENE_DUAL_PCT,BENE_RRL_PCT,BENE_AVG_AGE,BENE_MALE_PCT,BENE_FEML_PCT,BENE_RACE_WHT_PCT,BENE_RACE_BLACK_PCT,BENE_RACE_API_PCT,BENE_RACE_HSPNC_PCT,BENE_RACE_NATIND_PCT,BENE_RACE_UNK_PCT,BENE_RACE_OTHR_PCT,BENE_AVG_RISK_SCRE,BENE_CC_BH_ADHD_OTHCD_V1_PCT,BENE_CC_BH_ALCOHOL_DRUG_V1_PCT,BENE_CC_BH_ALZ_NONALZDEM_V2_PCT,BENE_CC_BH_ANXIETY_V1_PCT,BENE_CC_BH_BIPOLAR_V1_PCT,BENE_CC_BH_DEPRESS_V1_PCT,BENE_CC_BH_MOOD_V2_PCT,BENE_CC_BH_PD_V1_PCT,BENE_CC_BH_PTSD_V1_PCT,BENE_CC_BH_SCHIZO_OTHPSY_V1_PCT,BENE_CC_BH_TOBACCO_V1_PCT,BENE_CC_PH_AFIB_V2_PCT,BENE_CC_PH_ARTHRITIS_V2_PCT,BENE_CC_PH_ASTHMA_V2_PCT,BENE_CC_PH_CANCER6_V2_PCT,BENE_CC_PH_CKD_V2_PCT,BENE_CC_PH_COPD_V2_PCT,BENE_CC_PH_DIABETES_V2_PCT,BENE_CC_PH_HF_NONIHD_V2_PCT,BENE_CC_PH_HYPERLIPIDEMIA_V2_PCT,BENE_CC_PH_HYPERTENSION_V2_PCT,BENE_CC_PH_ISCHEMICHEART_V2_PCT,BENE_CC_PH_OSTEOPOROSIS_V2_PCT,BENE_CC_PH_PARKINSON_V2_PCT,BENE_CC_PH_STROKE_TIA_V2_PCT,PRMRY_DX_INFCTN_PCT,PRMRY_DX_NEOBLD_PCT,PRMRY_DX_ENDONUTRMET_PCT,PRMRY_DX_MNTBEHNEUDIS_PCT,PRMRY_DX_NERVSYSTM_PCT,PRMRY_DX_ENTSYS_PCT,PRMRY_DX_CIRCSYSTM_PCT,PRMRY_DX_RSPSYSTM_PCT,PRMRY_DX_DIGSYSTM_PCT,PRMRY_DX_SKNMUSSYSTM_PCT,PRMRY_DX_GUSYSTM_PCT,PRMRY_DX_PRGPERICONG_PCT,PRMRY_DX_SXILLDEF_PCT,PRMRY_DX_INJPOIS_PCT,PRMRY_DX_HLTHSRV_PCT,TOT_PT_MNTS,INDVDL_PT_MNTS,CNCRNT_GRP_PT_MNTS,COTRT_PT_MNTS,TOT_OT_MNTS,INDVDL_OT_MNTS,CNCRNT_GRP_OT_MNTS,COTRT_OT_MNTS,TOT_SLP_MNTS,INDVDL_SLP_MNTS,CNCRNT_GRP_SLP_MNTS,COTRT_SLP_MNTS
0,2024,FY,NATION,IRF,NATIONAL TOTAL,NATIONAL TOTAL,NATIONAL TOTAL,NATIONAL TOTAL,NATIONAL TOTAL,377403,434556,5348816,24958018679,11021437293,10822889016,10796423499,361242272,15,7,78,46,54,83,7,2,5,0,1,1,2.45,2,15,28,48,11,48,53,2,3,4,17,42,74,15,22,50,34,49,46,89,95,54,27,7,40,0,1,0,0,19,0,17,1,0,5,0,0,16,24,15,261638622,223450665,36196209,1991748,260369334,224479972,33891770,1997592,59054173,55378653,3509966,165554
1,2024,FY,STATE,IRF,STATE TOTAL,STATE TOTAL,STATE TOTAL,AK,STATE TOTAL,494,510,7336,70435069,16963613,16669709,14552147,3103105,21,21,75,51,49,74,*,4,2,13,*,4,1.85,*,17,19,32,7,31,33,*,3,*,22,39,69,14,18,35,26,41,36,77,91,40,23,5,53,*,*,*,*,9,0,32,*,0,3,*,0,9,30,14,294735,294174,163,398,277991,277241,0,750,73329,73056,90,183
2,2024,FY,STATE,IRF,STATE TOTAL,STATE TOTAL,STATE TOTAL,AL,STATE TOTAL,6919,7764,95695,303829686,167167535,163862709,196639061,2603753,13,7,78,44,56,84,14,0,1,*,1,*,2.58,1,14,32,48,10,47,52,2,2,4,20,46,76,15,21,56,37,53,50,90,98,57,23,6,39,0,0,0,*,25,*,16,1,0,3,0,*,14,24,15,5137615,3679451,1416378,41786,5168880,3775367,1352962,40551,1038623,941566,94385,2672
3,2024,FY,STATE,IRF,STATE TOTAL,STATE TOTAL,STATE TOTAL,AR,STATE TOTAL,10185,12006,140135,495887574,260441618,254243966,278520627,1299787,16,20,79,40,60,91,6,0,1,1,1,0,2.31,1,14,28,47,9,50,52,2,2,4,19,45,75,14,21,49,39,45,50,86,96,59,26,6,36,0,0,1,0,26,*,14,2,1,6,1,*,13,22,13,8041979,6310406,1639315,92258,7824938,6079521,1654635,90782,1735066,1420891,311296,2879
4,2024,FY,STATE,IRF,STATE TOTAL,STATE TOTAL,STATE TOTAL,AZ,STATE TOTAL,10951,12559,150913,598928960,306600356,300565428,296801567,3999086,10,3,78,46,54,84,2,1,7,3,1,1,2.25,1,17,27,44,9,46,50,2,3,3,14,41,73,16,22,49,31,43,38,88,94,51,31,8,40,0,1,0,0,17,*,16,1,0,6,0,0,13,29,15,6563426,5604310,900914,58202,6031765,5177650,795866,58249,1471926,1336511,134402,1013


In [4]:
# lets chop this up to National, state and provider level analysis 

df_raw["SMRY_CTGRY"].value_counts(dropna=False)

SMRY_CTGRY
PROVIDER    1165
STATE         52
NATION         1
Name: count, dtype: int64

In [5]:
df = df_raw[df_raw["SMRY_CTGRY"] == "PROVIDER"].copy()

print("Provider level rows:", df.shape[0])
print("Columns:", df.shape[1])

Provider level rows: 1165
Columns: 82


In [6]:
# column names and types of data 

column_inventory = pd.DataFrame({
    "Column": df.columns,
    "Data_Type": df.dtypes.astype(str).values
})

column_inventory

,Column,Data_Type
0,YEAR,int64
1,YEAR_TYPE,object
2,SMRY_CTGRY,object
3,SRVC_CTGRY,object
4,PRVDR_ID,object
5,PRVDR_NAME,object
6,PRVDR_CITY,object
7,STATE,object
8,PRVDR_ZIP,object
9,BENE_DSTNCT_CNT,int64


In [7]:
# lets disect this further 

audit = pd.DataFrame({
    "Column": df.columns,
    "Data_Type": df.dtypes.astype(str).values,
    "Missing_Count": df.isna().sum().values,
    "Missing_Percent": (df.isna().mean().values * 100).round(2),
    "Unique_Values": df.nunique(dropna=True).values
})

audit

,Column,Data_Type,Missing_Count,Missing_Percent,Unique_Values
0,YEAR,int64,0,0.0,1
1,YEAR_TYPE,object,0,0.0,1
2,SMRY_CTGRY,object,0,0.0,1
3,SRVC_CTGRY,object,0,0.0,1
4,PRVDR_ID,object,0,0.0,1165
5,PRVDR_NAME,object,0,0.0,1157
6,PRVDR_CITY,object,0,0.0,788
7,STATE,object,0,0.0,52
8,PRVDR_ZIP,object,0,0.0,1077
9,BENE_DSTNCT_CNT,int64,0,0.0,603


In [8]:
missing_summary = audit[audit["Missing_Count"] > 0].copy()

missing_summary = missing_summary.sort_values(
    "Missing_Percent",
    ascending=False
)

missing_summary

,Column,Data_Type,Missing_Count,Missing_Percent,Unique_Values


### Initial Missing-Value Assessment

No standard `NaN` or null values were identified in the provider level dataset.

However, this does not indicate complete data availability. CMS uses an asterisk (`*`) to represent values suppressed for privacy purposes. These suppressed values are stored as text and therefore are not automatically recognized by pandas as missing values.

CMS suppression will be evaluated separately before any variables are excluded or converted to numeric data types.

In [9]:
suppression_counts = {}

for col in df.columns:
    count = (df[col].astype(str).str.strip() == "*").sum()
    
    if count > 0:
        suppression_counts[col] = count

suppression_summary = pd.DataFrame(
    suppression_counts.items(),
    columns=["Column", "Suppressed_Count"]
)

suppression_summary["Suppressed_Percent"] = (
    suppression_summary["Suppressed_Count"] / len(df) * 100
).round(2)

suppression_summary = suppression_summary.sort_values(
    "Suppressed_Percent",
    ascending=False
)

suppression_summary

,Column,Suppressed_Count,Suppressed_Percent
9,BENE_RACE_UNK_PCT,874,75.02
11,BENE_CC_BH_ADHD_OTHCD_V1_PCT,806,69.18
18,BENE_CC_BH_PD_V1_PCT,762,65.41
10,BENE_RACE_OTHR_PCT,759,65.15
19,BENE_CC_BH_PTSD_V1_PCT,740,63.52
6,BENE_RACE_API_PCT,702,60.26
37,PRMRY_DX_NEOBLD_PCT,669,57.42
20,BENE_CC_BH_SCHIZO_OTHPSY_V1_PCT,626,53.73
7,BENE_RACE_HSPNC_PCT,612,52.53
31,BENE_CC_PH_HYPERTENSION_V2_PCT,587,50.39


In [10]:
# Lets disect the supression levels using numbers. 

def suppression_category(percent):
    if percent == 0:
        return "None"
    elif percent < 10:
        return "Low (<10%)"
    elif percent < 30:
        return "Moderate (10-30%)"
    elif percent < 50:
        return "High (30-50%)"
    else:
        return "Very High (>=50%)"

suppression_summary["Suppression_Level"] = (
    suppression_summary["Suppressed_Percent"]
    .apply(suppression_category)
)

suppression_summary

,Column,Suppressed_Count,Suppressed_Percent,Suppression_Level
9,BENE_RACE_UNK_PCT,874,75.02,Very High (>=50%)
11,BENE_CC_BH_ADHD_OTHCD_V1_PCT,806,69.18,Very High (>=50%)
18,BENE_CC_BH_PD_V1_PCT,762,65.41,Very High (>=50%)
10,BENE_RACE_OTHR_PCT,759,65.15,Very High (>=50%)
19,BENE_CC_BH_PTSD_V1_PCT,740,63.52,Very High (>=50%)
6,BENE_RACE_API_PCT,702,60.26,Very High (>=50%)
37,PRMRY_DX_NEOBLD_PCT,669,57.42,Very High (>=50%)
20,BENE_CC_BH_SCHIZO_OTHPSY_V1_PCT,626,53.73,Very High (>=50%)
7,BENE_RACE_HSPNC_PCT,612,52.53,Very High (>=50%)
31,BENE_CC_PH_HYPERTENSION_V2_PCT,587,50.39,Very High (>=50%)


In [11]:
# moving forward I am going to analyze missingness as ( Data Quality and analytical usefulness) not just % of missing data 

### CMS Suppression Assessment

CMS suppression varies substantially across variables in the provider level dataset.

Several race, behavioral health, chronic condition, and primary diagnosis variables show high or very high levels of suppression. Some variables have more than 50% of provider level values suppressed.

At this stage, variables are not being removed based solely on suppression percentage. Suppression level will be considered together with:

- Relevance to the provider anomaly analysis
- Whether suppression is associated with provider characteristics such as volume
- Whether the variable is needed for peer group comparisons
- Whether sufficient unsuppressed observations remain for meaningful analysis

Variables with substantial suppression will be evaluated individually before inclusion in later analyses.

In [12]:
print("Total variables:", len(df.columns))
print("Variables with CMS suppression:", len(suppression_summary))
print("Variables without CMS suppression:", len(df.columns) - len(suppression_summary))

print("\nSuppression levels:")
print(suppression_summary["Suppression_Level"].value_counts())


Total variables: 82
Variables with CMS suppression: 63
Variables without CMS suppression: 19

Suppression levels:
Suppression_Level
Low (<10%)           33
Very High (>=50%)    11
Moderate (10-30%)    10
High (30-50%)         9
Name: count, dtype: int64


In [13]:
# okay we can see how much is supressed but what are the variables? Lets build a variable map 

In [14]:
for i, col in enumerate(df.columns, start=1):
    print(f"{i:2}. {col}")

 1. YEAR
 2. YEAR_TYPE
 3. SMRY_CTGRY
 4. SRVC_CTGRY
 5. PRVDR_ID
 6. PRVDR_NAME
 7. PRVDR_CITY
 8. STATE
 9. PRVDR_ZIP
10. BENE_DSTNCT_CNT
11. TOT_EPSD_STAY_CNT
12. TOT_SRVC_DAYS
13. TOT_CHRG_AMT
14. TOT_ALOWD_AMT
15. TOT_MDCR_PYMT_AMT
16. TOT_MDCR_STDZD_PYMT_AMT
17. TOT_OUTLIER_PYMT_AMT
18. BENE_DUAL_PCT
19. BENE_RRL_PCT
20. BENE_AVG_AGE
21. BENE_MALE_PCT
22. BENE_FEML_PCT
23. BENE_RACE_WHT_PCT
24. BENE_RACE_BLACK_PCT
25. BENE_RACE_API_PCT
26. BENE_RACE_HSPNC_PCT
27. BENE_RACE_NATIND_PCT
28. BENE_RACE_UNK_PCT
29. BENE_RACE_OTHR_PCT
30. BENE_AVG_RISK_SCRE
31. BENE_CC_BH_ADHD_OTHCD_V1_PCT
32. BENE_CC_BH_ALCOHOL_DRUG_V1_PCT
33. BENE_CC_BH_ALZ_NONALZDEM_V2_PCT
34. BENE_CC_BH_ANXIETY_V1_PCT
35. BENE_CC_BH_BIPOLAR_V1_PCT
36. BENE_CC_BH_DEPRESS_V1_PCT
37. BENE_CC_BH_MOOD_V2_PCT
38. BENE_CC_BH_PD_V1_PCT
39. BENE_CC_BH_PTSD_V1_PCT
40. BENE_CC_BH_SCHIZO_OTHPSY_V1_PCT
41. BENE_CC_BH_TOBACCO_V1_PCT
42. BENE_CC_PH_AFIB_V2_PCT
43. BENE_CC_PH_ARTHRITIS_V2_PCT
44. BENE_CC_PH_ASTHMA_V2_PCT
45. BENE_C

In [15]:
# lets sort this 

def assign_variable_family(col):
    
    if col in ["YEAR", "YEAR_TYPE", "SMRY_CTGRY", "SRVC_CTGRY"]:
        return "Administrative"
    
    elif col in ["PRVDR_ID", "PRVDR_NAME", "PRVDR_CITY", "STATE", "PRVDR_ZIP"]:
        return "Provider / Geography"
    
    elif col in ["BENE_DSTNCT_CNT", "TOT_EPSD_STAY_CNT", "TOT_SRVC_DAYS"]:
        return "Utilization / Volume"
    
    elif col in [
        "TOT_CHRG_AMT",
        "TOT_ALOWD_AMT",
        "TOT_MDCR_PYMT_AMT",
        "TOT_MDCR_STDZD_PYMT_AMT",
        "TOT_OUTLIER_PYMT_AMT"
    ]:
        return "Payment / Charges"
    
    elif col.startswith("BENE_CC_"):
        return "Chronic Conditions"
    
    elif col.startswith("PRMRY_DX_"):
        return "Primary Diagnosis"
    
    elif any(x in col for x in ["PT_MNTS", "OT_MNTS", "SLP_MNTS"]):
        return "Therapy Utilization"
    
    elif col.startswith("BENE_"):
        return "Beneficiary Characteristics"
    
    else:
        return "Review"

variable_map = pd.DataFrame({
    "Column": df.columns
})

variable_map["Variable_Family"] = variable_map["Column"].apply(assign_variable_family)

variable_map

,Column,Variable_Family
0,YEAR,Administrative
1,YEAR_TYPE,Administrative
2,SMRY_CTGRY,Administrative
3,SRVC_CTGRY,Administrative
4,PRVDR_ID,Provider / Geography
5,PRVDR_NAME,Provider / Geography
6,PRVDR_CITY,Provider / Geography
7,STATE,Provider / Geography
8,PRVDR_ZIP,Provider / Geography
9,BENE_DSTNCT_CNT,Utilization / Volume


In [16]:
# combine 

def assign_variable_family(col):
    
    if col in ["YEAR", "YEAR_TYPE", "SMRY_CTGRY", "SRVC_CTGRY"]:
        return "Administrative"
    
    elif col in ["PRVDR_ID", "PRVDR_NAME", "PRVDR_CITY", "STATE", "PRVDR_ZIP"]:
        return "Provider / Geography"
    
    elif col in ["BENE_DSTNCT_CNT", "TOT_EPSD_STAY_CNT", "TOT_SRVC_DAYS"]:
        return "Utilization / Volume"
    
    elif col in [
        "TOT_CHRG_AMT",
        "TOT_ALOWD_AMT",
        "TOT_MDCR_PYMT_AMT",
        "TOT_MDCR_STDZD_PYMT_AMT",
        "TOT_OUTLIER_PYMT_AMT"
    ]:
        return "Payment / Charges"
    
    elif col.startswith("BENE_CC_"):
        return "Chronic Conditions"
    
    elif col.startswith("PRMRY_DX_"):
        return "Primary Diagnosis"
    
    elif any(x in col for x in ["PT_MNTS", "OT_MNTS", "SLP_MNTS"]):
        return "Therapy Utilization"
    
    elif col.startswith("BENE_"):
        return "Beneficiary Characteristics"
    
    else:
        return "Review"

variable_map = pd.DataFrame({
    "Column": df.columns
})

variable_map["Variable_Family"] = variable_map["Column"].apply(assign_variable_family)

variable_map

,Column,Variable_Family
0,YEAR,Administrative
1,YEAR_TYPE,Administrative
2,SMRY_CTGRY,Administrative
3,SRVC_CTGRY,Administrative
4,PRVDR_ID,Provider / Geography
5,PRVDR_NAME,Provider / Geography
6,PRVDR_CITY,Provider / Geography
7,STATE,Provider / Geography
8,PRVDR_ZIP,Provider / Geography
9,BENE_DSTNCT_CNT,Utilization / Volume


### Variable Review Criteria

Each variable will be evaluated based on its data quality, analytical relevance, and role in the provider anomaly analysis.

- **KEEP** — Needed for the analysis and may be used directly in calculations, comparisons, or modeling.

- **KEEP – ID/Context** — Useful for identifying providers, describing results, or creating comparison groups, but not intended to be used as a modeling variable.

- **REVIEW** — Potentially useful for the analysis, but requires further investigation due to suppression, data quality, interpretation, or other limitations.

- **EXCLUDE FROM ANALYSIS** — Not useful for the current analytical goal or does not provide meaningful variation for the planned analysis.

Variables marked for exclusion will not be deleted from the original dataset. They will remain available if needed later but will not be included in the analysis ready dataset unless their relevance changes.

In [17]:
master_audit = variable_map.merge(
    audit,
    on="Column",
    how="left"
)

master_audit = master_audit.merge(
    suppression_summary[
        ["Column", "Suppressed_Count", "Suppressed_Percent", "Suppression_Level"]
    ],
    on="Column",
    how="left"
)

master_audit["Suppressed_Count"] = (
    master_audit["Suppressed_Count"].fillna(0).astype(int)
)

master_audit["Suppressed_Percent"] = (
    master_audit["Suppressed_Percent"].fillna(0)
)

master_audit["Suppression_Level"] = (
    master_audit["Suppression_Level"].fillna("None")
)

master_audit

,Column,Variable_Family,Data_Type,Missing_Count,Missing_Percent,Unique_Values,Suppressed_Count,Suppressed_Percent,Suppression_Level
0,YEAR,Administrative,int64,0,0.0,1,0,0.00,None
1,YEAR_TYPE,Administrative,object,0,0.0,1,0,0.00,None
2,SMRY_CTGRY,Administrative,object,0,0.0,1,0,0.00,None
3,SRVC_CTGRY,Administrative,object,0,0.0,1,0,0.00,None
4,PRVDR_ID,Provider / Geography,object,0,0.0,1165,0,0.00,None
5,PRVDR_NAME,Provider / Geography,object,0,0.0,1157,0,0.00,None
6,PRVDR_CITY,Provider / Geography,object,0,0.0,788,0,0.00,None
7,STATE,Provider / Geography,object,0,0.0,52,0,0.00,None
8,PRVDR_ZIP,Provider / Geography,object,0,0.0,1077,0,0.00,None
9,BENE_DSTNCT_CNT,Utilization / Volume,int64,0,0.0,603,0,0.00,None


In [18]:
# lets start with Administrative 

admin_review = master_audit[
    master_audit["Variable_Family"] == "Administrative"
].copy()

admin_review

,Column,Variable_Family,Data_Type,Missing_Count,Missing_Percent,Unique_Values,Suppressed_Count,Suppressed_Percent,Suppression_Level
0,YEAR,Administrative,int64,0,0.0,1,0,0.0,None
1,YEAR_TYPE,Administrative,object,0,0.0,1,0,0.0,None
2,SMRY_CTGRY,Administrative,object,0,0.0,1,0,0.0,None
3,SRVC_CTGRY,Administrative,object,0,0.0,1,0,0.0,None


In [19]:
for col in ["YEAR", "YEAR_TYPE", "SMRY_CTGRY", "SRVC_CTGRY"]:
    print(f"{col}: {df[col].unique()}")

YEAR: [2024]
YEAR_TYPE: ['FY']
SMRY_CTGRY: ['PROVIDER']
SRVC_CTGRY: ['IRF']


In [20]:
# perfect. These are all constant across all 1,165 provider records 

In [21]:
# Create decision tracking columns
master_audit["Decision"] = ""
master_audit["Reason"] = ""

# Administrative variables are constant within the provider level dataset
admin_cols = ["YEAR", "YEAR_TYPE", "SMRY_CTGRY", "SRVC_CTGRY"]

master_audit.loc[
    master_audit["Column"].isin(admin_cols),
    "Decision"
] = "EXCLUDE FROM ANALYSIS"

master_audit.loc[
    master_audit["Column"].isin(admin_cols),
    "Reason"
] = "Constant across all provider records; retained as metadata but provides no variation for provider level analysis."

# Display the completed administrative review
master_audit[
    master_audit["Variable_Family"] == "Administrative"
][["Column", "Unique_Values", "Decision", "Reason"]]

,Column,Unique_Values,Decision,Reason
0,YEAR,1,EXCLUDE FROM ANALYSIS,Constant across all provider records; retained...
1,YEAR_TYPE,1,EXCLUDE FROM ANALYSIS,Constant across all provider records; retained...
2,SMRY_CTGRY,1,EXCLUDE FROM ANALYSIS,Constant across all provider records; retained...
3,SRVC_CTGRY,1,EXCLUDE FROM ANALYSIS,Constant across all provider records; retained...


In [22]:
provider_cols = [
    "PRVDR_ID",
    "PRVDR_NAME",
    "PRVDR_CITY",
    "STATE",
    "PRVDR_ZIP"
]

print("Total provider rows:", len(df))
print("Unique provider IDs:", df["PRVDR_ID"].nunique())
print("Duplicate provider IDs:", df["PRVDR_ID"].duplicated().sum())

print("\nUnique values:")
for col in provider_cols:
    print(f"{col}: {df[col].nunique()}")

Total provider rows: 1165
Unique provider IDs: 1165
Duplicate provider IDs: 0

Unique values:
PRVDR_ID: 1165
PRVDR_NAME: 1157
PRVDR_CITY: 788
STATE: 52
PRVDR_ZIP: 1077


In [23]:
# lets check pandas formatting and make sure they didn't screw things up 

print("Provider ID examples:")
print(df["PRVDR_ID"].head(10).tolist())

print("\nZIP code examples:")
print(df["PRVDR_ZIP"].head(10).tolist())

print("\nProvider ID data type:", df["PRVDR_ID"].dtype)
print("ZIP code data type:", df["PRVDR_ZIP"].dtype)

Provider ID examples:
['020017', '022001', '010011', '010033', '010092', '010100', '010104', '010113', '010114', '010139']

ZIP code examples:
['99508', '99503', '35235', '35233', '35401', '36532', '35243', '36607', '35022', '35216']

Provider ID data type: object
ZIP code data type: object


In [24]:
provider_geo_cols = [
    "PRVDR_ID",
    "PRVDR_NAME",
    "PRVDR_CITY",
    "STATE",
    "PRVDR_ZIP"
]

master_audit.loc[
    master_audit["Column"].isin(provider_geo_cols),
    "Decision"
] = "KEEP – ID/Context"

master_audit.loc[
    master_audit["Column"].isin(provider_geo_cols),
    "Reason"
] = (
    "Required for provider identification, geographic context, "
    "peer-group development, and interpretation of flagged providers; "
    "not intended as direct numeric modeling features."
)

master_audit[
    master_audit["Variable_Family"] == "Provider / Geography"
][["Column", "Data_Type", "Unique_Values", "Decision", "Reason"]]

,Column,Data_Type,Unique_Values,Decision,Reason
4,PRVDR_ID,object,1165,KEEP – ID/Context,"Required for provider identification, geograph..."
5,PRVDR_NAME,object,1157,KEEP – ID/Context,"Required for provider identification, geograph..."
6,PRVDR_CITY,object,788,KEEP – ID/Context,"Required for provider identification, geograph..."
7,STATE,object,52,KEEP – ID/Context,"Required for provider identification, geograph..."
8,PRVDR_ZIP,object,1077,KEEP – ID/Context,"Required for provider identification, geograph..."


In [25]:
utilization_cols = [
    "BENE_DSTNCT_CNT",
    "TOT_EPSD_STAY_CNT",
    "TOT_SRVC_DAYS"
]

master_audit[
    master_audit["Column"].isin(utilization_cols)
][
    [
        "Column",
        "Data_Type",
        "Unique_Values",
        "Suppressed_Count",
        "Suppressed_Percent",
        "Suppression_Level"
    ]
]

,Column,Data_Type,Unique_Values,Suppressed_Count,Suppressed_Percent,Suppression_Level
9,BENE_DSTNCT_CNT,int64,603,0,0.0,None
10,TOT_EPSD_STAY_CNT,int64,620,0,0.0,None
11,TOT_SRVC_DAYS,int64,1096,0,0.0,None


In [26]:
# zero suppressed, all int64, mo conversions needed. 

In [27]:
df[utilization_cols].describe().T

print("\nZero values:")
print((df[utilization_cols] == 0).sum())

print("\nNegative values:")
print((df[utilization_cols] < 0).sum())


Zero values:
BENE_DSTNCT_CNT      0
TOT_EPSD_STAY_CNT    0
TOT_SRVC_DAYS        0
dtype: int64

Negative values:
BENE_DSTNCT_CNT      0
TOT_EPSD_STAY_CNT    0
TOT_SRVC_DAYS        0
dtype: int64


In [28]:
# Zero zero's, zero negative values, -- KEEP 

### Utilization / Volume Review

The three utilization variables were retained for analysis:

- `BENE_DSTNCT_CNT` — distinct Medicare beneficiaries
- `TOT_EPSD_STAY_CNT` — total episodes/stays
- `TOT_SRVC_DAYS` — total service days

All three variables are numeric and contain no missing, CMS suppressed, zero, or negative values.

These variables will be retained because provider volume may be important when comparing facilities and interpreting payment variation. Raw utilization totals will not automatically be treated as outliers or data quality problems simply because values are extreme.

Additional normalized measures, such as average service days per stay, may be derived later to allow more meaningful comparisons between providers of different sizes.

In [29]:
# payment variables 

payment_cols = [
    "TOT_CHRG_AMT",
    "TOT_ALOWD_AMT",
    "TOT_MDCR_PYMT_AMT",
    "TOT_MDCR_STDZD_PYMT_AMT",
    "TOT_OUTLIER_PYMT_AMT"
]

master_audit[
    master_audit["Column"].isin(payment_cols)
][
    [
        "Column",
        "Data_Type",
        "Unique_Values",
        "Suppressed_Count",
        "Suppressed_Percent",
        "Suppression_Level"
    ]
]

,Column,Data_Type,Unique_Values,Suppressed_Count,Suppressed_Percent,Suppression_Level
12,TOT_CHRG_AMT,int64,1165,0,0.0,None
13,TOT_ALOWD_AMT,int64,1165,0,0.0,None
14,TOT_MDCR_PYMT_AMT,int64,1165,0,0.0,None
15,TOT_MDCR_STDZD_PYMT_AMT,int64,1165,0,0.0,None
16,TOT_OUTLIER_PYMT_AMT,int64,1066,0,0.0,None


In [30]:
# lets inspect actual values/ Distribution of payment and charge variables
df[payment_cols].describe().T


,count,mean,std,min,25%,50%,75%,max
TOT_CHRG_AMT,1165.0,2.142233e+07,2.577592e+07,224752.0,7286979.0,14672813.0,27390210.0,374279880.0
TOT_ALOWD_AMT,1165.0,9.459792e+06,9.613663e+06,165747.0,3076625.0,5833028.0,13206338.0,99259659.0
TOT_MDCR_PYMT_AMT,1165.0,9.289370e+06,9.418089e+06,161619.0,3034772.0,5761628.0,12954396.0,97583011.0
TOT_MDCR_STDZD_PYMT_AMT,1165.0,9.266730e+06,9.128799e+06,271105.0,3145899.0,5676481.0,12861917.0,81918824.0
TOT_OUTLIER_PYMT_AMT,1165.0,3.100659e+05,7.713997e+05,0.0,14389.0,77218.0,279819.0,9939920.0


In [31]:
print("\nZero values:")
print((df[payment_cols] == 0).sum())

print("\nNegative values:")
print((df[payment_cols] < 0).sum())


Zero values:
TOT_CHRG_AMT                0
TOT_ALOWD_AMT               0
TOT_MDCR_PYMT_AMT           0
TOT_MDCR_STDZD_PYMT_AMT     0
TOT_OUTLIER_PYMT_AMT       94
dtype: int64

Negative values:
TOT_CHRG_AMT               0
TOT_ALOWD_AMT              0
TOT_MDCR_PYMT_AMT          0
TOT_MDCR_STDZD_PYMT_AMT    0
TOT_OUTLIER_PYMT_AMT       0
dtype: int64


In [32]:
master_audit.loc[
    master_audit["Column"].isin(payment_cols),
    "Decision"
] = "KEEP"

master_audit.loc[
    master_audit["Column"].isin(payment_cols),
    "Reason"
] = (
    "Complete numeric payment/charge measure with no missing or suppressed values. "
    "Retained for payment analysis, normalization, provider comparison, "
    "and anomaly detection. Extreme values will be investigated rather than removed."
)

master_audit[
    master_audit["Column"].isin(payment_cols)
][
    [
        "Column",
        "Suppressed_Percent",
        "Decision",
        "Reason"
    ]
]

,Column,Suppressed_Percent,Decision,Reason
12,TOT_CHRG_AMT,0.0,KEEP,Complete numeric payment/charge measure with n...
13,TOT_ALOWD_AMT,0.0,KEEP,Complete numeric payment/charge measure with n...
14,TOT_MDCR_PYMT_AMT,0.0,KEEP,Complete numeric payment/charge measure with n...
15,TOT_MDCR_STDZD_PYMT_AMT,0.0,KEEP,Complete numeric payment/charge measure with n...
16,TOT_OUTLIER_PYMT_AMT,0.0,KEEP,Complete numeric payment/charge measure with n...


### Payment / Charge Variable Review

All five payment and charge variables were retained for analysis. They are numeric and contain no missing or CMS suppressed values.

The raw payment distributions are strongly right-skewed, with mean values substantially higher than median values and maximum values considerably above the upper quartile. These extreme values will not be removed during data cleaning because they may reflect legitimate differences in provider volume or may represent meaningful provider level variation requiring further investigation.

`TOT_OUTLIER_PYMT_AMT` contains 94 zero values. These values will be preserved because zero outlier payment may represent a valid provider level value rather than missing data.

Raw total payments will later be normalized using utilization measures, such as total stays, to improve comparisons between providers of different sizes.

In [33]:
beneficiary_review = master_audit[
    master_audit["Variable_Family"] == "Beneficiary Characteristics"
].copy()

beneficiary_review[
    [
        "Column",
        "Data_Type",
        "Unique_Values",
        "Suppressed_Count",
        "Suppressed_Percent",
        "Suppression_Level"
    ]
]

,Column,Data_Type,Unique_Values,Suppressed_Count,Suppressed_Percent,Suppression_Level
17,BENE_DUAL_PCT,object,76,168,14.42,Moderate (10-30%)
18,BENE_RRL_PCT,object,57,499,42.83,High (30-50%)
19,BENE_AVG_AGE,int64,18,0,0.00,None
20,BENE_MALE_PCT,object,44,23,1.97,Low (<10%)
21,BENE_FEML_PCT,object,44,23,1.97,Low (<10%)
22,BENE_RACE_WHT_PCT,object,77,19,1.63,Low (<10%)
23,BENE_RACE_BLACK_PCT,object,59,451,38.71,High (30-50%)
24,BENE_RACE_API_PCT,object,30,702,60.26,Very High (>=50%)
25,BENE_RACE_HSPNC_PCT,object,59,612,52.53,Very High (>=50%)
26,BENE_RACE_NATIND_PCT,object,17,334,28.67,Moderate (10-30%)


In [34]:
# dont change the df, just make a copy and work from there. 

beneficiary_cols = beneficiary_review["Column"].tolist()

beneficiary_numeric = df[beneficiary_cols].copy()

# Convert suppressed "*" values to NaN, then convert to numeric
for col in beneficiary_numeric.columns:
    beneficiary_numeric[col] = pd.to_numeric(
        beneficiary_numeric[col].replace("*", np.nan),
        errors="coerce"
    )

beneficiary_numeric.dtypes

BENE_DUAL_PCT           float64
BENE_RRL_PCT            float64
BENE_AVG_AGE              int64
BENE_MALE_PCT           float64
BENE_FEML_PCT           float64
BENE_RACE_WHT_PCT       float64
BENE_RACE_BLACK_PCT     float64
BENE_RACE_API_PCT       float64
BENE_RACE_HSPNC_PCT     float64
BENE_RACE_NATIND_PCT    float64
BENE_RACE_UNK_PCT       float64
BENE_RACE_OTHR_PCT      float64
BENE_AVG_RISK_SCRE      float64
dtype: object

In [35]:
# we are basically telling pandas to change the * values to supression as unavailable rather than just text. ONLY IN THE COPY NOT THE DF

In [36]:
# Percentage variables only
pct_cols = [
    col for col in beneficiary_numeric.columns
    if col.endswith("_PCT")
]

range_check = pd.DataFrame({
    "Min": beneficiary_numeric[pct_cols].min(),
    "Max": beneficiary_numeric[pct_cols].max(),
    "Values_Below_0": (beneficiary_numeric[pct_cols] < 0).sum(),
    "Values_Above_100": (beneficiary_numeric[pct_cols] > 100).sum(),
    "Available_Count": beneficiary_numeric[pct_cols].notna().sum(),
    "Missing_Count": beneficiary_numeric[pct_cols].isna().sum()
})

range_check

,Min,Max,Values_Below_0,Values_Above_100,Available_Count,Missing_Count
BENE_DUAL_PCT,0.0,87.0,0,0,989,176
BENE_RRL_PCT,0.0,91.0,0,0,666,499
BENE_MALE_PCT,26.0,71.0,0,0,1142,23
BENE_FEML_PCT,29.0,74.0,0,0,1142,23
BENE_RACE_WHT_PCT,0.0,100.0,0,0,1146,19
BENE_RACE_BLACK_PCT,0.0,89.0,0,0,714,451
BENE_RACE_API_PCT,0.0,76.0,0,0,463,702
BENE_RACE_HSPNC_PCT,0.0,100.0,0,0,553,612
BENE_RACE_NATIND_PCT,0.0,32.0,0,0,831,334
BENE_RACE_UNK_PCT,0.0,16.0,0,0,291,874


In [37]:
# supression value got higher? were non * variables changed to NaN? 

# Check original values that could not be converted to numeric
for col in beneficiary_cols:
    original = df[col]

    converted = pd.to_numeric(
        original.replace("*", np.nan),
        errors="coerce"
    )

    unexpected = original[
        converted.isna() & original.notna() & (original != "*")
    ]

    if len(unexpected) > 0:
        print(f"\n{col}")
        print("Unexpected non-numeric count:", len(unexpected))
        print("Values:", unexpected.value_counts())


BENE_DUAL_PCT
Unexpected non-numeric count: 8
Values: BENE_DUAL_PCT
+    8
Name: count, dtype: int64


In [38]:
# What the hell does + variable mean ? 

In [39]:
import os

for root, dirs, files in os.walk("data"):
    for file in files:
        print(os.path.join(root, file))

data/Medicare Post-Acute Care Utilization - Inpatient Rehabilitation Facility by Geography and Provider/2024/RY26_PACPUF_IRF_2024_main_S.csv


In [40]:
# Add a column for "+" special values -- data.cms.gov gave us some input on + values. 
master_audit["Special_Plus_Count"] = 0

master_audit.loc[
    master_audit["Column"] == "BENE_DUAL_PCT",
    "Special_Plus_Count"
] = 8

### CMS Special-Value Review

During numeric conversion of the beneficiary variables, 8 records in
`BENE_DUAL_PCT` contained the CMS special notation `+`.

These values are not numeric percentages and were therefore converted to
NaN in the numeric working copy. They are tracked separately from the 168
records containing `*` suppression notation.

As a result, `BENE_DUAL_PCT` has 989 usable numeric observations and
176 non-numeric observations (168 `*` values and 8 `+` values).

The original values remain unchanged in the raw/provider level dataset.

In [41]:
beneficiary_numeric[
    ["BENE_AVG_AGE", "BENE_AVG_RISK_SCRE"]
].describe().T

,count,mean,std,min,25%,50%,75%,max
BENE_AVG_AGE,1165.0,77.217167,2.266228,63.00,76.00,78.00,79.00,83.00
BENE_AVG_RISK_SCRE,1165.0,2.408807,0.447450,0.89,2.14,2.39,2.64,5.22


In [42]:
# lets record our findings, 

beneficiary_keep = [
    "BENE_AVG_AGE",
    "BENE_AVG_RISK_SCRE",
    "BENE_MALE_PCT",
    "BENE_FEML_PCT",
    "BENE_RACE_WHT_PCT",
    "BENE_DUAL_PCT"
]

beneficiary_review_cols = [
    "BENE_RRL_PCT",
    "BENE_RACE_BLACK_PCT",
    "BENE_RACE_NATIND_PCT",
    "BENE_RACE_API_PCT",
    "BENE_RACE_HSPNC_PCT",
    "BENE_RACE_UNK_PCT",
    "BENE_RACE_OTHR_PCT"
]

master_audit.loc[
    master_audit["Column"].isin(beneficiary_keep),
    "Decision"
] = "KEEP"

master_audit.loc[
    master_audit["Column"].isin(beneficiary_keep),
    "Reason"
] = (
    "Retained as beneficiary population/context measure. Numeric values passed "
    "range checks and the variable has sufficient usable observations for "
    "further analysis."
)

master_audit.loc[
    master_audit["Column"].isin(beneficiary_review_cols),
    "Decision"
] = "REVIEW"

master_audit.loc[
    master_audit["Column"].isin(beneficiary_review_cols),
    "Reason"
] = (
    "Potentially useful beneficiary context variable, but substantial CMS "
    "suppression requires further evaluation before use in provider comparisons "
    "or anomaly analysis."
)

In [43]:
# check
master_audit[
    master_audit["Variable_Family"] == "Beneficiary Characteristics"
][["Column", "Decision", "Reason"]]

,Column,Decision,Reason
17,BENE_DUAL_PCT,KEEP,Retained as beneficiary population/context mea...
18,BENE_RRL_PCT,REVIEW,Potentially useful beneficiary context variabl...
19,BENE_AVG_AGE,KEEP,Retained as beneficiary population/context mea...
20,BENE_MALE_PCT,KEEP,Retained as beneficiary population/context mea...
21,BENE_FEML_PCT,KEEP,Retained as beneficiary population/context mea...
22,BENE_RACE_WHT_PCT,KEEP,Retained as beneficiary population/context mea...
23,BENE_RACE_BLACK_PCT,REVIEW,Potentially useful beneficiary context variabl...
24,BENE_RACE_API_PCT,REVIEW,Potentially useful beneficiary context variabl...
25,BENE_RACE_HSPNC_PCT,REVIEW,Potentially useful beneficiary context variabl...
26,BENE_RACE_NATIND_PCT,REVIEW,Potentially useful beneficiary context variabl...


In [44]:
# lets do a supression count per provider so far. (*)  

beneficiary_suppression_count = (
    df[beneficiary_cols]
    .astype(str)
    .apply(lambda col: col.str.strip().eq("*"))
    .sum(axis=1)
)

suppression_volume_check = pd.DataFrame({
    "PRVDR_ID": df["PRVDR_ID"],
    "BENE_DSTNCT_CNT": df["BENE_DSTNCT_CNT"],
    "TOT_EPSD_STAY_CNT": df["TOT_EPSD_STAY_CNT"],
    "Suppressed_Beneficiary_Variables": beneficiary_suppression_count
})

suppression_volume_check.head()

,PRVDR_ID,BENE_DSTNCT_CNT,TOT_EPSD_STAY_CNT,Suppressed_Beneficiary_Variables
53,020017,120,122,5
54,022001,375,388,3
55,010011,46,48,3
56,010033,151,164,4
57,010092,243,263,3


In [45]:
# suppression vs. provider volume 

suppression_volume_check[
    [
        "BENE_DSTNCT_CNT",
        "TOT_EPSD_STAY_CNT",
        "Suppressed_Beneficiary_Variables"
    ]
].corr()

,BENE_DSTNCT_CNT,TOT_EPSD_STAY_CNT,Suppressed_Beneficiary_Variables
BENE_DSTNCT_CNT,1.000000,0.998677,-0.175332
TOT_EPSD_STAY_CNT,0.998677,1.000000,-0.176952
Suppressed_Beneficiary_Variables,-0.175332,-0.176952,1.000000


This is telling us that as provider volume increases, supression decreases. negative correlation. So the question still stands. Do the smallest 25% of providers actually have more suppressed beneficiary variables than the largest 25%?

In [46]:
suppression_volume_check["Volume_Group"] = pd.qcut(
    suppression_volume_check["BENE_DSTNCT_CNT"],
    q=4,
    labels=[
        "Lowest Volume",
        "Low-Mid Volume",
        "High-Mid Volume",
        "Highest Volume"
    ]
)

suppression_by_volume = (
    suppression_volume_check
    .groupby("Volume_Group", observed=True)
    ["Suppressed_Beneficiary_Variables"]
    .agg(["count", "mean", "median", "min", "max"])
)

suppression_by_volume

,count,mean,median,min,max
Volume_Group,,,,,
Lowest Volume,295,3.857627,4.0,0,9
Low-Mid Volume,290,3.896552,4.0,0,7
High-Mid Volume,290,4.065517,4.0,2,7
Highest Volume,290,3.506897,4.0,0,7


### Beneficiary Suppression vs. Provider Volume

CMS suppression was evaluated in relation to provider volume using beneficiary
count and total episode/stay count.

Suppression showed a weak negative correlation with both beneficiary count
(r = -0.175) and total episode/stay count (r = -0.177), indicating that
higher volume providers tend to have fewer suppressed beneficiary
variables.

However, comparison across beneficiary volume quartiles did not show a clear
stepwise relationship. Median suppression was 4 variables in all four volume
groups, and mean suppression was similar across the lowest, low/mid, and
high/mid volume groups. The highest volume group had somewhat lower mean
suppression (3.51 variables).

Therefore, provider volume may contribute to suppression patterns but does
not independently explain them. Variables with substantial suppression will
remain flagged for review rather than excluding providers or variables solely
based on suppression.

In [47]:
# chronic conditions (*) deep dive 

chronic_review = master_audit[
    master_audit["Variable_Family"] == "Chronic Conditions"
].copy()

chronic_review[
    [
        "Column",
        "Data_Type",
        "Unique_Values",
        "Suppressed_Count",
        "Suppressed_Percent",
        "Suppression_Level"
    ]
]

,Column,Data_Type,Unique_Values,Suppressed_Count,Suppressed_Percent,Suppression_Level
30,BENE_CC_BH_ADHD_OTHCD_V1_PCT,object,11,806,69.18,Very High (>=50%)
31,BENE_CC_BH_ALCOHOL_DRUG_V1_PCT,object,36,154,13.22,Moderate (10-30%)
32,BENE_CC_BH_ALZ_NONALZDEM_V2_PCT,object,47,65,5.58,Low (<10%)
33,BENE_CC_BH_ANXIETY_V1_PCT,object,55,33,2.83,Low (<10%)
34,BENE_CC_BH_BIPOLAR_V1_PCT,object,39,383,32.88,High (30-50%)
35,BENE_CC_BH_DEPRESS_V1_PCT,object,51,32,2.75,Low (<10%)
36,BENE_CC_BH_MOOD_V2_PCT,object,58,29,2.49,Low (<10%)
37,BENE_CC_BH_PD_V1_PCT,object,17,762,65.41,Very High (>=50%)
38,BENE_CC_BH_PTSD_V1_PCT,object,12,740,63.52,Very High (>=50%)
39,BENE_CC_BH_SCHIZO_OTHPSY_V1_PCT,object,28,626,53.73,Very High (>=50%)


In [48]:
# this is a mess. nothing is uniform. Lets fix this. Make a copy to work with and keep primary df alone. 

chronic_cols = chronic_review["Column"].tolist()

chronic_numeric = df[chronic_cols].copy()

# Convert CMS "*" suppression values to NaN
for col in chronic_numeric.columns:
    chronic_numeric[col] = pd.to_numeric(
        chronic_numeric[col].replace("*", np.nan),
        errors="coerce"
    )

chronic_numeric.dtypes

BENE_CC_BH_ADHD_OTHCD_V1_PCT        float64
BENE_CC_BH_ALCOHOL_DRUG_V1_PCT      float64
BENE_CC_BH_ALZ_NONALZDEM_V2_PCT     float64
BENE_CC_BH_ANXIETY_V1_PCT           float64
BENE_CC_BH_BIPOLAR_V1_PCT           float64
BENE_CC_BH_DEPRESS_V1_PCT           float64
BENE_CC_BH_MOOD_V2_PCT              float64
BENE_CC_BH_PD_V1_PCT                float64
BENE_CC_BH_PTSD_V1_PCT              float64
BENE_CC_BH_SCHIZO_OTHPSY_V1_PCT     float64
BENE_CC_BH_TOBACCO_V1_PCT           float64
BENE_CC_PH_AFIB_V2_PCT              float64
BENE_CC_PH_ARTHRITIS_V2_PCT         float64
BENE_CC_PH_ASTHMA_V2_PCT            float64
BENE_CC_PH_CANCER6_V2_PCT           float64
BENE_CC_PH_CKD_V2_PCT               float64
BENE_CC_PH_COPD_V2_PCT              float64
BENE_CC_PH_DIABETES_V2_PCT          float64
BENE_CC_PH_HF_NONIHD_V2_PCT         float64
BENE_CC_PH_HYPERLIPIDEMIA_V2_PCT    float64
BENE_CC_PH_HYPERTENSION_V2_PCT      float64
BENE_CC_PH_ISCHEMICHEART_V2_PCT     float64
BENE_CC_PH_OSTEOPOROSIS_V2_PCT  

In [49]:
# lets check if theres anything other than (*) 

for col in chronic_cols:
    original = df[col]

    converted = pd.to_numeric(
        original.replace("*", np.nan),
        errors="coerce"
    )

    unexpected = original[
        converted.isna() &
        original.notna() &
        (original.astype(str).str.strip() != "*")
    ]

    if len(unexpected) > 0:
        print(f"\n{col}")
        print("Unexpected non-numeric count:", len(unexpected))
        print("Values:")
        print(unexpected.value_counts())

In [50]:
# 0-100 check 

chronic_range_check = pd.DataFrame({
    "Min": chronic_numeric.min(),
    "Max": chronic_numeric.max(),
    "Values_Below_0": (chronic_numeric < 0).sum(),
    "Values_Above_100": (chronic_numeric > 100).sum(),
    "Available_Count": chronic_numeric.notna().sum(),
    "Missing_Count": chronic_numeric.isna().sum()
})

chronic_range_check

,Min,Max,Values_Below_0,Values_Above_100,Available_Count,Missing_Count
BENE_CC_BH_ADHD_OTHCD_V1_PCT,0.0,18.0,0,0,359,806
BENE_CC_BH_ALCOHOL_DRUG_V1_PCT,0.0,54.0,0,0,1011,154
BENE_CC_BH_ALZ_NONALZDEM_V2_PCT,11.0,65.0,0,0,1100,65
BENE_CC_BH_ANXIETY_V1_PCT,16.0,87.0,0,0,1132,33
BENE_CC_BH_BIPOLAR_V1_PCT,0.0,62.0,0,0,782,383
BENE_CC_BH_DEPRESS_V1_PCT,0.0,84.0,0,0,1133,32
BENE_CC_BH_MOOD_V2_PCT,16.0,92.0,0,0,1136,29
BENE_CC_BH_PD_V1_PCT,0.0,34.0,0,0,403,762
BENE_CC_BH_PTSD_V1_PCT,0.0,14.0,0,0,425,740
BENE_CC_BH_SCHIZO_OTHPSY_V1_PCT,0.0,67.0,0,0,539,626


In [51]:
chronic_review_cols = [
    "BENE_CC_BH_ADHD_OTHCD_V1_PCT",
    "BENE_CC_BH_BIPOLAR_V1_PCT",
    "BENE_CC_BH_PD_V1_PCT",
    "BENE_CC_BH_PTSD_V1_PCT",
    "BENE_CC_BH_SCHIZO_OTHPSY_V1_PCT",
    "BENE_CC_PH_HYPERTENSION_V2_PCT",
    "BENE_CC_PH_PARKINSON_V2_PCT"
]

chronic_keep_cols = [
    col for col in chronic_cols
    if col not in chronic_review_cols
]

master_audit.loc[
    master_audit["Column"].isin(chronic_keep_cols),
    "Decision"
] = "KEEP"

master_audit.loc[
    master_audit["Column"].isin(chronic_keep_cols),
    "Reason"
] = (
    "Valid percentage measure with sufficient provider-level availability. "
    "Retained as potential patient complexity/context information for further analysis."
)

master_audit.loc[
    master_audit["Column"].isin(chronic_review_cols),
    "Decision"
] = "REVIEW"

master_audit.loc[
    master_audit["Column"].isin(chronic_review_cols),
    "Reason"
] = (
    "Valid percentage measure, but substantial CMS suppression (>=30%) "
    "limits provider level availability and requires further evaluation before analysis."
)

In [52]:
master_audit[
    master_audit["Variable_Family"] == "Chronic Conditions"
][
    [
        "Column",
        "Suppressed_Percent",
        "Decision",
        "Reason"
    ]
]

,Column,Suppressed_Percent,Decision,Reason
30,BENE_CC_BH_ADHD_OTHCD_V1_PCT,69.18,REVIEW,"Valid percentage measure, but substantial CMS ..."
31,BENE_CC_BH_ALCOHOL_DRUG_V1_PCT,13.22,KEEP,Valid percentage measure with sufficient provi...
32,BENE_CC_BH_ALZ_NONALZDEM_V2_PCT,5.58,KEEP,Valid percentage measure with sufficient provi...
33,BENE_CC_BH_ANXIETY_V1_PCT,2.83,KEEP,Valid percentage measure with sufficient provi...
34,BENE_CC_BH_BIPOLAR_V1_PCT,32.88,REVIEW,"Valid percentage measure, but substantial CMS ..."
35,BENE_CC_BH_DEPRESS_V1_PCT,2.75,KEEP,Valid percentage measure with sufficient provi...
36,BENE_CC_BH_MOOD_V2_PCT,2.49,KEEP,Valid percentage measure with sufficient provi...
37,BENE_CC_BH_PD_V1_PCT,65.41,REVIEW,"Valid percentage measure, but substantial CMS ..."
38,BENE_CC_BH_PTSD_V1_PCT,63.52,REVIEW,"Valid percentage measure, but substantial CMS ..."
39,BENE_CC_BH_SCHIZO_OTHPSY_V1_PCT,53.73,REVIEW,"Valid percentage measure, but substantial CMS ..."


### Chronic Condition Variable Review

All chronic condition % variables were evaluated for numeric validity,
CMS suppression, and provider level availability. All available numeric values
fell within the expected 0–100% range, and no unexpected non-numeric values
were identified beyond the CMS `*` suppression.

Variables with less than 30% suppression were retained (KEEP) because sufficient
provider level data remain available for continued analysis.

Variables with 30% or greater suppression were flagged as REVIEW rather than
excluded. The 30% threshold is used here as a practical data quality screening
rule for this analysis, not as a CMS defined cutoff. These variables may still
contain useful information, but their reduced provider coverage could affect
provider comparisons or anomaly detection.

KEEP indicates that a variable is suitable for continued consideration during
analysis; it does not mean the variable will necessarily be included in the
final anomaly detection model.

In [53]:
# lets check inventory list of Primary DX

primary_dx_review = master_audit[
    master_audit["Variable_Family"] == "Primary Diagnosis"
].copy()

primary_dx_review[
    [
        "Column",
        "Data_Type",
        "Unique_Values",
        "Suppressed_Count",
        "Suppressed_Percent",
        "Suppression_Level"
    ]
]

,Column,Data_Type,Unique_Values,Suppressed_Count,Suppressed_Percent,Suppression_Level
55,PRMRY_DX_INFCTN_PCT,object,9,490,42.06,High (30-50%)
56,PRMRY_DX_NEOBLD_PCT,object,9,669,57.42,Very High (>=50%)
57,PRMRY_DX_ENDONUTRMET_PCT,object,7,575,49.36,High (30-50%)
58,PRMRY_DX_MNTBEHNEUDIS_PCT,object,5,250,21.46,Moderate (10-30%)
59,PRMRY_DX_NERVSYSTM_PCT,object,56,288,24.72,Moderate (10-30%)
60,PRMRY_DX_ENTSYS_PCT,object,2,149,12.79,Moderate (10-30%)
61,PRMRY_DX_CIRCSYSTM_PCT,object,61,75,6.44,Low (<10%)
62,PRMRY_DX_RSPSYSTM_PCT,object,18,585,50.21,Very High (>=50%)
63,PRMRY_DX_DIGSYSTM_PCT,object,6,580,49.79,High (30-50%)
64,PRMRY_DX_SKNMUSSYSTM_PCT,object,30,533,45.75,High (30-50%)


In [54]:
# copy 

primary_dx_cols = primary_dx_review["Column"].tolist()

primary_dx_numeric = df[primary_dx_cols].copy()  # copy, leaving df alone. 

# changing CMS "*" suppression values to NaN
for col in primary_dx_numeric.columns:
    primary_dx_numeric[col] = pd.to_numeric(
        primary_dx_numeric[col].replace("*", np.nan),
        errors="coerce"
    )

primary_dx_numeric.dtypes

PRMRY_DX_INFCTN_PCT          float64
PRMRY_DX_NEOBLD_PCT          float64
PRMRY_DX_ENDONUTRMET_PCT     float64
PRMRY_DX_MNTBEHNEUDIS_PCT    float64
PRMRY_DX_NERVSYSTM_PCT       float64
PRMRY_DX_ENTSYS_PCT          float64
PRMRY_DX_CIRCSYSTM_PCT       float64
PRMRY_DX_RSPSYSTM_PCT        float64
PRMRY_DX_DIGSYSTM_PCT        float64
PRMRY_DX_SKNMUSSYSTM_PCT     float64
PRMRY_DX_GUSYSTM_PCT         float64
PRMRY_DX_PRGPERICONG_PCT     float64
PRMRY_DX_SXILLDEF_PCT        float64
PRMRY_DX_INJPOIS_PCT         float64
PRMRY_DX_HLTHSRV_PCT         float64
dtype: object

In [55]:
# check for other other supression symbols 

for col in primary_dx_cols:
    original = df[col]

    converted = pd.to_numeric(
        original.replace("*", np.nan),
        errors="coerce"
    )

    unexpected = original[
        converted.isna() &
        original.notna() &
        (original.astype(str).str.strip() != "*")
    ]

    if len(unexpected) > 0:
        print(f"\n{col}")
        print("Unexpected non-numeric count:", len(unexpected))
        print("Values:")
        print(unexpected.value_counts())

In [56]:
primary_dx_range_check = pd.DataFrame({
    "Min": primary_dx_numeric.min(),
    "Max": primary_dx_numeric.max(),
    "Values_Below_0": (primary_dx_numeric < 0).sum(),
    "Values_Above_100": (primary_dx_numeric > 100).sum(),
    "Available_Count": primary_dx_numeric.notna().sum(),
    "Missing_Count": primary_dx_numeric.isna().sum()
})

primary_dx_range_check

,Min,Max,Values_Below_0,Values_Above_100,Available_Count,Missing_Count
PRMRY_DX_INFCTN_PCT,0.0,9.0,0,0,675,490
PRMRY_DX_NEOBLD_PCT,0.0,15.0,0,0,496,669
PRMRY_DX_ENDONUTRMET_PCT,0.0,5.0,0,0,590,575
PRMRY_DX_MNTBEHNEUDIS_PCT,0.0,3.0,0,0,915,250
PRMRY_DX_NERVSYSTM_PCT,0.0,59.0,0,0,877,288
PRMRY_DX_ENTSYS_PCT,0.0,0.0,0,0,1016,149
PRMRY_DX_CIRCSYSTM_PCT,0.0,71.0,0,0,1090,75
PRMRY_DX_RSPSYSTM_PCT,0.0,25.0,0,0,580,585
PRMRY_DX_DIGSYSTM_PCT,0.0,4.0,0,0,585,580
PRMRY_DX_SKNMUSSYSTM_PCT,0.0,55.0,0,0,632,533


In [57]:
# based on 30% supression rule I created, heres what we have 

primary_dx_review_cols = [
    "PRMRY_DX_INFCTN_PCT",
    "PRMRY_DX_NEOBLD_PCT",
    "PRMRY_DX_ENDONUTRMET_PCT",
    "PRMRY_DX_RSPSYSTM_PCT",
    "PRMRY_DX_DIGSYSTM_PCT",
    "PRMRY_DX_SKNMUSSYSTM_PCT",
    "PRMRY_DX_GUSYSTM_PCT"
]

primary_dx_exclude_cols = [
    "PRMRY_DX_ENTSYS_PCT",
    "PRMRY_DX_PRGPERICONG_PCT"
]

primary_dx_keep_cols = [
    col for col in primary_dx_cols
    if col not in primary_dx_review_cols
    and col not in primary_dx_exclude_cols
]

# KEEP
master_audit.loc[
    master_audit["Column"].isin(primary_dx_keep_cols),
    "Decision"
] = "KEEP"

master_audit.loc[
    master_audit["Column"].isin(primary_dx_keep_cols),
    "Reason"
] = (
    "Valid percentage measure with sufficient provider level availability "
    "and meaningful variation; retained for further analysis."
)

# REVIEW
master_audit.loc[
    master_audit["Column"].isin(primary_dx_review_cols),
    "Decision"
] = "REVIEW"

master_audit.loc[
    master_audit["Column"].isin(primary_dx_review_cols),
    "Reason"
] = (
    "Valid percentage measure, but substantial CMS suppression (>=30%) "
    "limits provider level availability and requires further evaluation."
)

# EXCLUDE
master_audit.loc[
    master_audit["Column"].isin(primary_dx_exclude_cols),
    "Decision"
] = "EXCLUDE FROM ANALYSIS"

master_audit.loc[
    master_audit["Column"].isin(primary_dx_exclude_cols),
    "Reason"
] = (
    "Available numeric values are constant at 0%, providing no variation "
    "for provider level comparison or anomaly detection."
)

In [58]:
master_audit[
    master_audit["Variable_Family"] == "Primary Diagnosis"
][
    ["Column", "Suppressed_Percent", "Decision", "Reason"]
]

,Column,Suppressed_Percent,Decision,Reason
55,PRMRY_DX_INFCTN_PCT,42.06,REVIEW,"Valid percentage measure, but substantial CMS ..."
56,PRMRY_DX_NEOBLD_PCT,57.42,REVIEW,"Valid percentage measure, but substantial CMS ..."
57,PRMRY_DX_ENDONUTRMET_PCT,49.36,REVIEW,"Valid percentage measure, but substantial CMS ..."
58,PRMRY_DX_MNTBEHNEUDIS_PCT,21.46,KEEP,Valid percentage measure with sufficient provi...
59,PRMRY_DX_NERVSYSTM_PCT,24.72,KEEP,Valid percentage measure with sufficient provi...
60,PRMRY_DX_ENTSYS_PCT,12.79,EXCLUDE FROM ANALYSIS,"Available numeric values are constant at 0%, p..."
61,PRMRY_DX_CIRCSYSTM_PCT,6.44,KEEP,Valid percentage measure with sufficient provi...
62,PRMRY_DX_RSPSYSTM_PCT,50.21,REVIEW,"Valid percentage measure, but substantial CMS ..."
63,PRMRY_DX_DIGSYSTM_PCT,49.79,REVIEW,"Valid percentage measure, but substantial CMS ..."
64,PRMRY_DX_SKNMUSSYSTM_PCT,45.75,REVIEW,"Valid percentage measure, but substantial CMS ..."


### Primary Diagnosis Variable Review

All primary diagnosis variables were converted to numeric format after treating
CMS `*` suppression values as unavailable. No additional unexpected non-numeric
values were identified, and all reported percentages were within the expected
0–100% range.

Variables with less than 30% suppression and meaningful variation were retained
(KEEP). Variables with 30% or greater suppression were flagged as REVIEW due to
limited provider level availability.

Two variables (`PRMRY_DX_ENTSYS_PCT` and `PRMRY_DX_PRGPERICONG_PCT`) were
excluded from analysis because all available numeric values were 0%, providing
no variation for provider level comparison or anomaly detection.

Suppressed variables remain available in the original dataset and may be
reconsidered if needed.

In [59]:
# therapy utilization (OT, PT, and SLP

therapy_review = master_audit[
    master_audit["Variable_Family"] == "Therapy Utilization"
].copy()

therapy_review[
    [
        "Column",
        "Data_Type",
        "Unique_Values",
        "Suppressed_Count",
        "Suppressed_Percent",
        "Suppression_Level"
    ]
]

,Column,Data_Type,Unique_Values,Suppressed_Count,Suppressed_Percent,Suppression_Level
70,TOT_PT_MNTS,object,1157,6,0.52,Low (<10%)
71,INDVDL_PT_MNTS,object,1158,6,0.52,Low (<10%)
72,CNCRNT_GRP_PT_MNTS,object,710,6,0.52,Low (<10%)
73,COTRT_PT_MNTS,object,444,6,0.52,Low (<10%)
74,TOT_OT_MNTS,object,1159,6,0.52,Low (<10%)
75,INDVDL_OT_MNTS,object,1158,6,0.52,Low (<10%)
76,CNCRNT_GRP_OT_MNTS,object,721,6,0.52,Low (<10%)
77,COTRT_OT_MNTS,object,440,6,0.52,Low (<10%)
78,TOT_SLP_MNTS,object,1154,6,0.52,Low (<10%)
79,INDVDL_SLP_MNTS,object,1151,6,0.52,Low (<10%)


In [60]:
# lets make this uniform 

therapy_cols = therapy_review["Column"].tolist()

therapy_numeric = df[therapy_cols].copy()

# Convert CMS "*" suppression values to NaN
for col in therapy_numeric.columns:
    therapy_numeric[col] = pd.to_numeric(
        therapy_numeric[col].replace("*", np.nan),
        errors="coerce"
    )

therapy_numeric.dtypes

TOT_PT_MNTS            float64
INDVDL_PT_MNTS         float64
CNCRNT_GRP_PT_MNTS     float64
COTRT_PT_MNTS          float64
TOT_OT_MNTS            float64
INDVDL_OT_MNTS         float64
CNCRNT_GRP_OT_MNTS     float64
COTRT_OT_MNTS          float64
TOT_SLP_MNTS           float64
INDVDL_SLP_MNTS        float64
CNCRNT_GRP_SLP_MNTS    float64
COTRT_SLP_MNTS         float64
dtype: object

In [61]:
# any weird symbols? 

for col in therapy_cols:
    original = df[col]

    converted = pd.to_numeric(
        original.replace("*", np.nan),
        errors="coerce"
    )

    unexpected = original[
        converted.isna() &
        original.notna() &
        (original.astype(str).str.strip() != "*")
    ]

    if len(unexpected) > 0:
        print(f"\n{col}")
        print("Unexpected non-numeric count:", len(unexpected))
        print("Values:")
        print(unexpected.value_counts())

In [62]:
therapy_quality_check = pd.DataFrame({
    "Min": therapy_numeric.min(),
    "Max": therapy_numeric.max(),
    "Zero_Count": (therapy_numeric == 0).sum(),
    "Negative_Count": (therapy_numeric < 0).sum(),
    "Available_Count": therapy_numeric.notna().sum(),
    "Missing_Count": therapy_numeric.isna().sum()
})

therapy_quality_check

,Min,Max,Zero_Count,Negative_Count,Available_Count,Missing_Count
TOT_PT_MNTS,6821.0,1558125.0,0,0,1159,6
INDVDL_PT_MNTS,5624.0,1491246.0,0,0,1159,6
CNCRNT_GRP_PT_MNTS,0.0,530417.0,356,0,1159,6
COTRT_PT_MNTS,0.0,80945.0,506,0,1159,6
TOT_OT_MNTS,6090.0,1614430.0,0,0,1159,6
INDVDL_OT_MNTS,5927.0,1382112.0,0,0,1159,6
CNCRNT_GRP_OT_MNTS,0.0,548198.0,331,0,1159,6
COTRT_OT_MNTS,0.0,83402.0,520,0,1159,6
TOT_SLP_MNTS,30.0,550874.0,0,0,1159,6
INDVDL_SLP_MNTS,0.0,529986.0,1,0,1159,6


In [63]:
# across all therapy types, are the supressed providers the same ones across all therapy types? 

therapy_suppression_by_provider = (
    df[therapy_cols]
    .astype(str)
    .apply(lambda col: col.str.strip().eq("*"))
)

therapy_suppression_counts = (
    therapy_suppression_by_provider
    .sum(axis=1)
    .value_counts()
    .sort_index()
)

therapy_suppression_counts

0     1159
12       6
Name: count, dtype: int64

### Therapy Utilization Suppression Pattern

Therapy utilization data showed very limited CMS suppression. Of the 1,165
provider level records, 1,159 providers had complete data for all 12 therapy
variables.

The remaining 6 providers had all 12 therapy variables suppressed. No providers
had partial suppression within the therapy utilization family.

Therefore, therapy suppression appears to occur at the provider level rather
than independently across individual therapy measures. These 6 providers will
remain in the dataset, but therapy based analyses will require consideration
of their unavailable therapy data.

In [64]:
# Lets calculate the difference (Total minutes versus individual + concurrent/group + co-treatment minutes)

therapy_component_check = pd.DataFrame({
    "PT_Difference": (
        therapy_numeric["TOT_PT_MNTS"]
        - (
            therapy_numeric["INDVDL_PT_MNTS"]
            + therapy_numeric["CNCRNT_GRP_PT_MNTS"]
            + therapy_numeric["COTRT_PT_MNTS"]
        )
    ),
    
    "OT_Difference": (
        therapy_numeric["TOT_OT_MNTS"]
        - (
            therapy_numeric["INDVDL_OT_MNTS"]
            + therapy_numeric["CNCRNT_GRP_OT_MNTS"]
            + therapy_numeric["COTRT_OT_MNTS"]
        )
    ),
    
    "SLP_Difference": (
        therapy_numeric["TOT_SLP_MNTS"]
        - (
            therapy_numeric["INDVDL_SLP_MNTS"]
            + therapy_numeric["CNCRNT_GRP_SLP_MNTS"]
            + therapy_numeric["COTRT_SLP_MNTS"]
        )
    )
})

therapy_component_check.describe().T

,count,mean,std,min,25%,50%,75%,max
PT_Difference,1159.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
OT_Difference,1159.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
SLP_Difference,1159.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [65]:
# NICE! this is good. 

master_audit.loc[
    master_audit["Column"].isin(therapy_cols),
    "Decision"
] = "KEEP"

master_audit.loc[
    master_audit["Column"].isin(therapy_cols),
    "Reason"
] = (
    "Valid therapy utilization measure with low CMS suppression (0.52%), "
    "no negative values, and complete internal consistency between total and "
    "component therapy minutes; retained for provider comparison and feature engineering."
)

master_audit[
    master_audit["Variable_Family"] == "Therapy Utilization"
][
    ["Column", "Suppressed_Percent", "Decision", "Reason"]
]

,Column,Suppressed_Percent,Decision,Reason
70,TOT_PT_MNTS,0.52,KEEP,Valid therapy utilization measure with low CMS...
71,INDVDL_PT_MNTS,0.52,KEEP,Valid therapy utilization measure with low CMS...
72,CNCRNT_GRP_PT_MNTS,0.52,KEEP,Valid therapy utilization measure with low CMS...
73,COTRT_PT_MNTS,0.52,KEEP,Valid therapy utilization measure with low CMS...
74,TOT_OT_MNTS,0.52,KEEP,Valid therapy utilization measure with low CMS...
75,INDVDL_OT_MNTS,0.52,KEEP,Valid therapy utilization measure with low CMS...
76,CNCRNT_GRP_OT_MNTS,0.52,KEEP,Valid therapy utilization measure with low CMS...
77,COTRT_OT_MNTS,0.52,KEEP,Valid therapy utilization measure with low CMS...
78,TOT_SLP_MNTS,0.52,KEEP,Valid therapy utilization measure with low CMS...
79,INDVDL_SLP_MNTS,0.52,KEEP,Valid therapy utilization measure with low CMS...


### Therapy Utilization Variable Review

All 12 therapy utilization variables were evaluated for CMS suppression,
unexpected non-numeric values, negative values, zero values, and internal
consistency.

Therapy data were available for 1,159 of 1,165 providers. The remaining
6 providers had all 12 therapy variables suppressed; no providers showed
partial suppression within this variable family.

No negative therapy minute values were identified. Zero values occurred
primarily in concurrent/group and co-treatment measures and were retained
as valid reported values.

For every provider with available therapy data, total PT, OT, and SLP minutes
exactly equaled the sum of individual, concurrent/group, and co-treatment
minutes for the corresponding discipline.

All 12 therapy utilization variables were therefore retained (KEEP).
These measures may later support feature engineering such as therapy minutes
per stay and the proportion of therapy delivered through different treatment
formats.

In [66]:
# all variable families have been looked at, lets look at the master copy real quick 

master_audit["Decision"].value_counts(dropna=False)

Decision
KEEP                     47
REVIEW                   21
EXCLUDE FROM ANALYSIS     6
KEEP – ID/Context         5
                          3
Name: count, dtype: int64

In [67]:
# we missed 3.....

master_audit[
    master_audit["Decision"] == ""
][
    [
        "Column",
        "Variable_Family",
        "Data_Type",
        "Suppressed_Percent",
        "Suppression_Level"
    ]
]

,Column,Variable_Family,Data_Type,Suppressed_Percent,Suppression_Level
9,BENE_DSTNCT_CNT,Utilization / Volume,int64,0.0,None
10,TOT_EPSD_STAY_CNT,Utilization / Volume,int64,0.0,None
11,TOT_SRVC_DAYS,Utilization / Volume,int64,0.0,None


In [68]:
# we already did this but it didn't save to the copy. Check. 

utilization_cols = [
    "BENE_DSTNCT_CNT",
    "TOT_EPSD_STAY_CNT",
    "TOT_SRVC_DAYS"
]

master_audit.loc[
    master_audit["Column"].isin(utilization_cols),
    "Decision"
] = "KEEP"

master_audit.loc[
    master_audit["Column"].isin(utilization_cols),
    "Reason"
] = (
    "Complete numeric utilization measure with no missing, suppressed, "
    "zero, or negative values; relevant for provider volume, peer comparisons, "
    "normalization, and feature engineering."
)

master_audit["Decision"].value_counts(dropna=False)

Decision
KEEP                     50
REVIEW                   21
EXCLUDE FROM ANALYSIS     6
KEEP – ID/Context         5
Name: count, dtype: int64

In [69]:
# nice. all accounted for now. Lets look at final decision and review....

final_decision_review = master_audit[
    master_audit["Decision"].isin([
        "REVIEW",
        "EXCLUDE FROM ANALYSIS"
    ])
][
    [
        "Column",
        "Variable_Family",
        "Suppressed_Percent",
        "Decision",
        "Reason"
    ]
].sort_values(
    ["Decision", "Variable_Family", "Suppressed_Percent"],
    ascending=[True, True, False]
)

final_decision_review

,Column,Variable_Family,Suppressed_Percent,Decision,Reason
0,YEAR,Administrative,0.00,EXCLUDE FROM ANALYSIS,Constant across all provider records; retained...
1,YEAR_TYPE,Administrative,0.00,EXCLUDE FROM ANALYSIS,Constant across all provider records; retained...
2,SMRY_CTGRY,Administrative,0.00,EXCLUDE FROM ANALYSIS,Constant across all provider records; retained...
3,SRVC_CTGRY,Administrative,0.00,EXCLUDE FROM ANALYSIS,Constant across all provider records; retained...
60,PRMRY_DX_ENTSYS_PCT,Primary Diagnosis,12.79,EXCLUDE FROM ANALYSIS,"Available numeric values are constant at 0%, p..."
66,PRMRY_DX_PRGPERICONG_PCT,Primary Diagnosis,8.15,EXCLUDE FROM ANALYSIS,"Available numeric values are constant at 0%, p..."
27,BENE_RACE_UNK_PCT,Beneficiary Characteristics,75.02,REVIEW,Potentially useful beneficiary context variabl...
28,BENE_RACE_OTHR_PCT,Beneficiary Characteristics,65.15,REVIEW,Potentially useful beneficiary context variabl...
24,BENE_RACE_API_PCT,Beneficiary Characteristics,60.26,REVIEW,Potentially useful beneficiary context variabl...
25,BENE_RACE_HSPNC_PCT,Beneficiary Characteristics,52.53,REVIEW,Potentially useful beneficiary context variabl...


## Final Variable Audit Summary

All 82 variables in the provider level dataset were reviewed for analytical
relevance, data type, missingness, CMS suppression, valid value ranges,
variation, and internal consistency where applicable.

Final classification:

- **50 KEEP** — suitable for continued analysis and feature engineering
- **5 KEEP – ID/Context** — retained for provider identification, geography,
  peer grouping, and interpretation
- **21 REVIEW** — potentially useful but limited primarily by substantial
  CMS suppression and requiring additional consideration before modeling
- **6 EXCLUDE FROM ANALYSIS** — administrative constants or variables with
  no meaningful numeric variation

Variables classified as REVIEW or EXCLUDE remain available in the original
provider level dataset and have not been deleted.

The original dataset (`df_raw`) and provider level dataset (`df`) remain
unchanged. Numeric working copies were created only for data quality
assessment.

## Feature Engineering

Following completion of the variable level data quality audit, derived
provider level features were created to normalize utilization and payment
measures.

Normalization is important because raw totals are influenced by provider
volume. Derived measures such as service days per stay allow providers of
different sizes to be compared more meaningfully.

### Average Service Days per Stay

Average service days per stay was calculated as:

TOT_SRVC_DAYS / TOT_EPSD_STAY_CNT

This represents the average number of service days associated with each
episode/stay for a provider.

In [70]:
# feature engineering 

df["AVG_SRVC_DAYS_PER_STAY"] = (
    df["TOT_SRVC_DAYS"] / df["TOT_EPSD_STAY_CNT"]
)

df["AVG_SRVC_DAYS_PER_STAY"].describe()

count    1165.000000
mean       12.444793
std         1.650336
min         8.153846
25%        11.436170
50%        12.245150
75%        13.266272
max        22.888889
Name: AVG_SRVC_DAYS_PER_STAY, dtype: float64

### Initial Interpretation

Across 1,165 IRF providers, the average service days per stay was approximately
12.44 days, with a median of 12.25 days.

The middle 50% of providers ranged from approximately 11.44 to 13.27 service
days per stay. Values ranged from 8.15 to 22.89 days per stay.

Higher or lower values are not assumed to represent inappropriate utilization.
These values will be evaluated later using provider peer comparisons and
anomaly detection methods.

In [71]:
# lets show the math. This equation is important. 

df["STDZD_PYMT_PER_STAY"] = (
    df["TOT_MDCR_STDZD_PYMT_AMT"] / df["TOT_EPSD_STAY_CNT"]
)

df["STDZD_PYMT_PER_STAY"].describe().round(2)

count     1165.00
mean     25526.16
std       4371.53
min      18330.90
25%      23203.83
50%      24552.98
75%      26414.32
max      64822.60
Name: STDZD_PYMT_PER_STAY, dtype: float64

### Standardized Medicare Payment per Stay

Standardized Medicare payment per stay was calculated as:

TOT_MDCR_STDZD_PYMT_AMT / TOT_EPSD_STAY_CNT

Across 1,165 providers, the mean standardized Medicare payment per stay was
approximately $25,526, with a median of approximately $24,553.

The middle 50% of providers ranged from approximately $23,204 to $26,414
per stay. Values ranged from approximately $18,331 to $64,823 per stay.

The upper end of the distribution contains substantially higher values;
however, these values are not classified as anomalies based on descriptive
statistics alone. They will be evaluated later using provider peer comparisons
and anomaly detection methods.

In [72]:
df["MDCR_PYMT_PER_STAY"] = (
    df["TOT_MDCR_PYMT_AMT"] / df["TOT_EPSD_STAY_CNT"]
)

df["MDCR_PYMT_PER_STAY"].describe().round(2)

count     1165.00
mean     25685.81
std       6768.81
min      10101.19
25%      21808.43
50%      23954.22
75%      27184.86
max      79672.33
Name: MDCR_PYMT_PER_STAY, dtype: float64

### Medicare Payment per Stay

Medicare payment per stay was calculated as:

TOT_MDCR_PYMT_AMT / TOT_EPSD_STAY_CNT

Across 1,165 providers, the mean Medicare payment per stay was approximately
$25,685.81, with a median of $23,954.22.

The middle 50% of providers ranged from approximately $21,808.43 to
$27,184.86 per stay. Values ranged from $10,101.19 to $79,672.33 per stay.

The wider range compared with standardized Medicare payment suggests that
actual and standardized payments may provide different information when
comparing providers. Extreme values will be evaluated later using appropriate
peer comparisons and anomaly detection methods.

In [73]:
# outlier analysis 

df["OUTLIER_PYMT_PER_STAY"] = (
    df["TOT_OUTLIER_PYMT_AMT"] / df["TOT_EPSD_STAY_CNT"]
)

df["OUTLIER_PYMT_PER_STAY"].describe().round(2)

count     1165.00
mean      1603.70
std       3586.87
min          0.00
25%         49.89
50%        348.30
75%       1334.33
max      36951.38
Name: OUTLIER_PYMT_PER_STAY, dtype: float64

### CMS Outlier Payment per Stay

CMS outlier payment per stay was calculated as:

TOT_OUTLIER_PYMT_AMT / TOT_EPSD_STAY_CNT

Across 1,165 providers, the mean CMS outlier payment per stay was approximately
$1,603.70, while the median was substantially lower at $348.30.

The middle 50% of providers ranged from approximately $49.89 to $1,334.33
per stay. Values ranged from $0.00 to $36,951.38 per stay.

The substantial difference between the mean and median, along with the large
upper range, suggests a right-skewed distribution that warrants further
evaluation during provider comparison and anomaly detection.

CMS outlier payments represent a specific Medicare payment category and should
not be interpreted as indicating that a provider is a statistical outlier or
engaged in fraud, waste, or abuse.

In [74]:
# allowed amount per stay (PER STAY is key word) 

df["ALOWD_AMT_PER_STAY"] = (
    df["TOT_ALOWD_AMT"] / df["TOT_EPSD_STAY_CNT"]
)

df["ALOWD_AMT_PER_STAY"].describe().round(2)

count     1165.00
mean     26099.83
std       6789.89
min      10292.63
25%      22261.76
50%      24383.89
75%      27638.58
max      80175.64
Name: ALOWD_AMT_PER_STAY, dtype: float64

### Allowed Amount per Stay

Allowed amount per stay was calculated as:

TOT_ALOWD_AMT / TOT_EPSD_STAY_CNT

Across 1,165 providers, the mean allowed amount per stay was approximately
$26,099.83, with a median of $24,383.89.

The middle 50% of providers ranged from approximately $22,261.76 to
$27,638.58 per stay. Values ranged from $10,292.63 to $80,175.64 per stay.

The upper range contains substantially higher values than the median.
These values are retained for later provider level comparison and
anomaly detection analysis rather than being removed during data preparation.

Observing now, investigate later. 

In [75]:
df["CHRG_AMT_PER_STAY"] = (
    df["TOT_CHRG_AMT"] / df["TOT_EPSD_STAY_CNT"]
)

df["CHRG_AMT_PER_STAY"].describe().round(2)

count      1165.00
mean      66427.93
std       41543.92
min       14131.99
25%       36493.74
50%       53098.64
75%       81518.41
max      314725.37
Name: CHRG_AMT_PER_STAY, dtype: float64

In [76]:
# this is showing total charges billed by providers, not what was paid. 

### Charges per Stay

Charges per stay was calculated as:

TOT_CHRG_AMT / TOT_EPSD_STAY_CNT

Across 1,165 providers, the mean charge per stay was approximately $66,427.93,
with a median of $53,098.64.

The middle 50% of providers ranged from approximately $36,493.74 to
$81,518.41 per stay. Values ranged from $14,131.99 to $314,725.37 per stay.

Charges per stay showed a wider distribution than the Medicare payment and
allowed amount measures. Because provider charges do not represent the amount
ultimately allowed or paid by Medicare, this measure will be interpreted
separately from payment measures.

Higher values will be retained for later provider comparison rather than
treated as errors or inappropriate billing during data preparation.

In [77]:
# lets lool at all of these engineered features side by side. 

engineered_features = [
    "AVG_SRVC_DAYS_PER_STAY",
    "STDZD_PYMT_PER_STAY",
    "MDCR_PYMT_PER_STAY",
    "OUTLIER_PYMT_PER_STAY",
    "ALOWD_AMT_PER_STAY",
    "CHRG_AMT_PER_STAY"
]

df[engineered_features].describe().round(2).T

,count,mean,std,min,25%,50%,75%,max
AVG_SRVC_DAYS_PER_STAY,1165.0,12.44,1.65,8.15,11.44,12.25,13.27,22.89
STDZD_PYMT_PER_STAY,1165.0,25526.16,4371.53,18330.90,23203.83,24552.98,26414.32,64822.60
MDCR_PYMT_PER_STAY,1165.0,25685.81,6768.81,10101.19,21808.43,23954.22,27184.86,79672.33
OUTLIER_PYMT_PER_STAY,1165.0,1603.70,3586.87,0.00,49.89,348.30,1334.33,36951.38
ALOWD_AMT_PER_STAY,1165.0,26099.83,6789.89,10292.63,22261.76,24383.89,27638.58,80175.64
CHRG_AMT_PER_STAY,1165.0,66427.93,41543.92,14131.99,36493.74,53098.64,81518.41,314725.37


In [78]:
# therapy utilization 

df["PT_MNTS_PER_STAY"] = (
    pd.to_numeric(df["TOT_PT_MNTS"].replace("*", np.nan), errors="coerce")
    / df["TOT_EPSD_STAY_CNT"]
)

df["PT_MNTS_PER_STAY"].describe().round(2)

count    1159.00
mean      597.71
std       135.91
min        37.80
25%       535.73
50%       616.41
75%       684.41
max      1072.76
Name: PT_MNTS_PER_STAY, dtype: float64

### Physical Therapy Minutes per Stay

Physical therapy minutes per stay was calculated as:

TOT_PT_MNTS / TOT_EPSD_STAY_CNT

Among the 1,159 providers with available therapy data, the mean total PT
minutes per stay was approximately 597.71 minutes, with a median of
616.41 minutes.

The middle 50% of providers ranged from approximately 535.73 to 684.41
minutes per stay. Values ranged from 37.80 to 1,072.76 minutes per stay.

Six providers had CMS suppressed therapy data and were retained as missing
rather than being interpreted as zero therapy utilization.

Low and high values are retained for later provider comparison and
anomaly detection analysis.

In [79]:
# OT same logic

df["OT_MNTS_PER_STAY"] = (
    pd.to_numeric(df["TOT_OT_MNTS"].replace("*", np.nan), errors="coerce")
    / df["TOT_EPSD_STAY_CNT"]
)

df["OT_MNTS_PER_STAY"].describe().round(2)

count    1159.00
mean      591.39
std       137.18
min        36.75
25%       527.43
50%       613.66
75%       678.68
max       967.43
Name: OT_MNTS_PER_STAY, dtype: float64

### Occupational Therapy Minutes per Stay

Occupational therapy minutes per stay was calculated as:

TOT_OT_MNTS / TOT_EPSD_STAY_CNT

Among the 1,159 providers with available therapy data, the mean total OT
minutes per stay was approximately 591.39 minutes, with a median of
613.66 minutes.

The middle 50% of providers ranged from approximately 527.43 to 678.68
minutes per stay. Values ranged from 36.75 to 967.43 minutes per stay.

Six providers had CMS suppressed therapy data and remain missing rather
than being interpreted as zero therapy utilization. Low and high values
are retained for later provider comparison and anomaly detection analysis.

In [80]:
#SLP same thing 

df["SLP_MNTS_PER_STAY"] = (
    pd.to_numeric(df["TOT_SLP_MNTS"].replace("*", np.nan), errors="coerce")
    / df["TOT_EPSD_STAY_CNT"]
)

df["SLP_MNTS_PER_STAY"].describe().round(2)

count    1159.00
mean      141.83
std        81.24
min         1.15
25%        83.27
50%       128.04
75%       187.94
max       616.81
Name: SLP_MNTS_PER_STAY, dtype: float64

### Speech-Language Pathology Minutes per Stay

Speech/language pathology (SLP) minutes per stay was calculated as:

TOT_SLP_MNTS / TOT_EPSD_STAY_CNT

Among the 1,159 providers with available therapy data, the mean total SLP
minutes per stay was approximately 141.83 minutes, with a median of
128.04 minutes.

The middle 50% of providers ranged from approximately 83.27 to 187.94
minutes per stay. Values ranged from 1.15 to 616.81 minutes per stay.

Six providers had CMS suppressed therapy data and remain missing rather
than being interpreted as zero therapy utilization. Variation in SLP
utilization may reflect differences in patient populations and will be
evaluated later using appropriate provider comparisons.

In [81]:
# lets run these 3 side by side 

therapy_features = [
    "PT_MNTS_PER_STAY",
    "OT_MNTS_PER_STAY",
    "SLP_MNTS_PER_STAY"
]

df[therapy_features].describe().round(2).T

,count,mean,std,min,25%,50%,75%,max
PT_MNTS_PER_STAY,1159.0,597.71,135.91,37.80,535.73,616.41,684.41,1072.76
OT_MNTS_PER_STAY,1159.0,591.39,137.18,36.75,527.43,613.66,678.68,967.43
SLP_MNTS_PER_STAY,1159.0,141.83,81.24,1.15,83.27,128.04,187.94,616.81


In [82]:
# we are going to analyze therapy individually/co therapy treatments and mixed therapy starting with PT 

df["PT_INDVDL_PCT"] = (
    pd.to_numeric(df["INDVDL_PT_MNTS"].replace("*", np.nan), errors="coerce")
    / pd.to_numeric(df["TOT_PT_MNTS"].replace("*", np.nan), errors="coerce")
    * 100
)

df["PT_INDVDL_PCT"].describe().round(2)

count    1159.00
mean       90.89
std        11.60
min        32.09
25%        84.59
50%        96.78
75%        99.92
max       100.00
Name: PT_INDVDL_PCT, dtype: float64

### Individual Physical Therapy Percentage

Individual PT percentage was calculated as:

INDVDL_PT_MNTS / TOT_PT_MNTS * 100

Among the 1,159 providers with available therapy data, the mean percentage
of PT minutes delivered individually was approximately 90.89%, with a
median of 96.78%.

The middle 50% of providers ranged from approximately 84.59% to 99.92%
individual PT. Values ranged from 32.09% to 100%.

This feature describes how PT services are delivered rather than the total
amount of PT provided. Variation in therapy delivery patterns will be
retained for later provider comparison and pattern discovery analysis.

In [83]:
# concurrent treatment info 

df["PT_CNCRNT_GRP_PCT"] = (
    pd.to_numeric(df["CNCRNT_GRP_PT_MNTS"].replace("*", np.nan), errors="coerce")
    / pd.to_numeric(df["TOT_PT_MNTS"].replace("*", np.nan), errors="coerce")
    * 100
)

df["PT_COTRT_PCT"] = (
    pd.to_numeric(df["COTRT_PT_MNTS"].replace("*", np.nan), errors="coerce")
    / pd.to_numeric(df["TOT_PT_MNTS"].replace("*", np.nan), errors="coerce")
    * 100
)

df[
    ["PT_INDVDL_PCT", "PT_CNCRNT_GRP_PCT", "PT_COTRT_PCT"]
].describe().round(2).T

,count,mean,std,min,25%,50%,75%,max
PT_INDVDL_PCT,1159.0,90.89,11.60,32.09,84.59,96.78,99.92,100.00
PT_CNCRNT_GRP_PCT,1159.0,7.69,11.24,0.00,0.00,1.13,12.41,59.25
PT_COTRT_PCT,1159.0,1.42,4.47,0.00,0.00,0.02,0.59,67.58


In [84]:
# 100% ? 

df["PT_PCT_TOTAL_CHECK"] = (
    df["PT_INDVDL_PCT"]
    + df["PT_CNCRNT_GRP_PCT"]
    + df["PT_COTRT_PCT"]
)

df["PT_PCT_TOTAL_CHECK"].describe().round(2)

count    1159.0
mean      100.0
std         0.0
min       100.0
25%       100.0
50%       100.0
75%       100.0
max       100.0
Name: PT_PCT_TOTAL_CHECK, dtype: float64

### Physical Therapy Delivery Pattern

PT delivery was represented as the percentage of total PT minutes provided
through individual therapy, concurrent/group therapy, and co-treatment.

Across providers, individual PT was the predominant delivery method, with a
median of 96.78% of total PT minutes. Concurrent/group PT had a median of
1.13%, while co-treatment had a median of 0.02%.

Substantial variation was present across providers, particularly in the upper
ranges of concurrent/group and co-treatment percentages. These differences
represent provider therapy delivery patterns and will be retained for later
peer comparison and pattern discovery analysis.

As a quality control check, the three engineered PT percentages summed to
exactly 100% for all 1,159 providers with available therapy data, confirming
consistency with the original PT component measures.

In [85]:
# OT in one clean block like we did for PT 

# Convert OT variables to numeric
ot_total = pd.to_numeric(
    df["TOT_OT_MNTS"].replace("*", np.nan),
    errors="coerce"
)

ot_individual = pd.to_numeric(
    df["INDVDL_OT_MNTS"].replace("*", np.nan),
    errors="coerce"
)

ot_concurrent = pd.to_numeric(
    df["CNCRNT_GRP_OT_MNTS"].replace("*", np.nan),
    errors="coerce"
)

ot_cotreatment = pd.to_numeric(
    df["COTRT_OT_MNTS"].replace("*", np.nan),
    errors="coerce"
)

# Create OT delivery percentages
df["OT_INDVDL_PCT"] = (ot_individual / ot_total) * 100
df["OT_CNCRNT_GRP_PCT"] = (ot_concurrent / ot_total) * 100
df["OT_COTRT_PCT"] = (ot_cotreatment / ot_total) * 100

df[
    ["OT_INDVDL_PCT", "OT_CNCRNT_GRP_PCT", "OT_COTRT_PCT"]
].describe().round(2).T

,count,mean,std,min,25%,50%,75%,max
OT_INDVDL_PCT,1159.0,91.52,10.66,29.19,85.72,96.57,99.87,100.00
OT_CNCRNT_GRP_PCT,1159.0,7.05,10.17,0.00,0.00,1.32,11.46,58.90
OT_COTRT_PCT,1159.0,1.43,4.50,0.00,0.00,0.02,0.55,67.66


In [86]:
# quality check

df["OT_PCT_TOTAL_CHECK"] = (
    df["OT_INDVDL_PCT"]
    + df["OT_CNCRNT_GRP_PCT"]
    + df["OT_COTRT_PCT"]
)

df["OT_PCT_TOTAL_CHECK"].describe().round(2)

count    1159.0
mean      100.0
std         0.0
min       100.0
25%       100.0
50%       100.0
75%       100.0
max       100.0
Name: OT_PCT_TOTAL_CHECK, dtype: float64

### Occupational Therapy Delivery Pattern

OT delivery was represented as the percentage of total OT minutes provided
through individual therapy, concurrent/group therapy, and co-treatment.

Individual OT was the predominant delivery method, with a median of 96.57%.
Concurrent/group OT had a median of 1.32%, while co-treatment had a median
of 0.02%.

Variation was present across providers, particularly at the upper ranges of
concurrent/group and co-treatment percentages. These differences were retained
as potential provider practice pattern features for later comparison and
pattern discovery analysis.

As a quality control check, the three OT delivery percentages summed to
exactly 100% for all 1,159 providers with available therapy data.

In [87]:
# SLP same process

# Convert SLP variables to numeric
slp_total = pd.to_numeric(
    df["TOT_SLP_MNTS"].replace("*", np.nan),
    errors="coerce"
)

slp_individual = pd.to_numeric(
    df["INDVDL_SLP_MNTS"].replace("*", np.nan),
    errors="coerce"
)

slp_concurrent = pd.to_numeric(
    df["CNCRNT_GRP_SLP_MNTS"].replace("*", np.nan),
    errors="coerce"
)

slp_cotreatment = pd.to_numeric(
    df["COTRT_SLP_MNTS"].replace("*", np.nan),
    errors="coerce"
)

# Create SLP percentages
df["SLP_INDVDL_PCT"] = (slp_individual / slp_total) * 100
df["SLP_CNCRNT_GRP_PCT"] = (slp_concurrent / slp_total) * 100
df["SLP_COTRT_PCT"] = (slp_cotreatment / slp_total) * 100

df[
    ["SLP_INDVDL_PCT", "SLP_CNCRNT_GRP_PCT", "SLP_COTRT_PCT"]
].describe().round(2).T

,count,mean,std,min,25%,50%,75%,max
SLP_INDVDL_PCT,1159.0,96.46,7.39,0.0,96.6,99.83,100.00,100.00
SLP_CNCRNT_GRP_PCT,1159.0,3.18,7.15,0.0,0.0,0.00,2.48,100.00
SLP_COTRT_PCT,1159.0,0.36,1.70,0.0,0.0,0.00,0.02,23.69


In [88]:
# quality check 

df["SLP_PCT_TOTAL_CHECK"] = (
    df["SLP_INDVDL_PCT"]
    + df["SLP_CNCRNT_GRP_PCT"]
    + df["SLP_COTRT_PCT"]
)

df["SLP_PCT_TOTAL_CHECK"].describe().round(2)

count    1159.0
mean      100.0
std         0.0
min       100.0
25%       100.0
50%       100.0
75%       100.0
max       100.0
Name: SLP_PCT_TOTAL_CHECK, dtype: float64

### Speech/Language Pathology Delivery Pattern

SLP delivery was represented as the percentage of total SLP minutes provided
through individual therapy, concurrent/group therapy, and co-treatment.

Individual SLP was the predominant delivery method, with a median of 99.83%.
Concurrent/group and co-treatment SLP both had median values of 0%.

Greater variation was observed among a smaller number of providers, including
providers with substantially higher concurrent/group therapy percentages.
These values are retained as potential provider practice pattern features and
will be interpreted alongside overall SLP utilization rather than independently.

As a quality control check, the three SLP delivery percentages summed to
exactly 100% for all 1,159 providers with available therapy data.

In [89]:
# lets look at all engineered features so far 

engineered_features = [
    "AVG_SRVC_DAYS_PER_STAY",
    "STDZD_PYMT_PER_STAY",
    "MDCR_PYMT_PER_STAY",
    "OUTLIER_PYMT_PER_STAY",
    "ALOWD_AMT_PER_STAY",
    "CHRG_AMT_PER_STAY",
    
    "PT_MNTS_PER_STAY",
    "OT_MNTS_PER_STAY",
    "SLP_MNTS_PER_STAY",
    
    "PT_INDVDL_PCT",
    "PT_CNCRNT_GRP_PCT",
    "PT_COTRT_PCT",
    
    "OT_INDVDL_PCT",
    "OT_CNCRNT_GRP_PCT",
    "OT_COTRT_PCT",
    
    "SLP_INDVDL_PCT",
    "SLP_CNCRNT_GRP_PCT",
    "SLP_COTRT_PCT"
]

feature_inventory = pd.DataFrame({
    "Feature": engineered_features,
    "Data_Type": df[engineered_features].dtypes.astype(str).values,
    "Available_Count": df[engineered_features].count().values,
    "Missing_Count": df[engineered_features].isna().sum().values
})

feature_inventory

,Feature,Data_Type,Available_Count,Missing_Count
0,AVG_SRVC_DAYS_PER_STAY,float64,1165,0
1,STDZD_PYMT_PER_STAY,float64,1165,0
2,MDCR_PYMT_PER_STAY,float64,1165,0
3,OUTLIER_PYMT_PER_STAY,float64,1165,0
4,ALOWD_AMT_PER_STAY,float64,1165,0
5,CHRG_AMT_PER_STAY,float64,1165,0
6,PT_MNTS_PER_STAY,float64,1159,6
7,OT_MNTS_PER_STAY,float64,1159,6
8,SLP_MNTS_PER_STAY,float64,1159,6
9,PT_INDVDL_PCT,float64,1159,6


In [90]:
df_clean = df.copy()

df_clean.shape

(1165, 103)

In [91]:
# claude code suggestion to clean things up a bit 

# Combine the percentage variable groups already identified
numeric_clean_cols = (
    beneficiary_cols
    + chronic_cols
    + primary_dx_cols
)

# Convert CMS special values to missing and remaining values to numeric
for col in numeric_clean_cols:
    df_clean[col] = pd.to_numeric(
        df_clean[col].replace(["*", "+"], np.nan),
        errors="coerce"
    )

df_clean[numeric_clean_cols].dtypes.value_counts()

float64    52
int64       1
Name: count, dtype: int64

In [92]:
# validate conversion

cleaning_validation = pd.DataFrame({
    "Column": numeric_clean_cols,
    "Missing_After_Cleaning": [
        df_clean[col].isna().sum()
        for col in numeric_clean_cols
    ],
    "Original_Star_Count": [
        (df[col].astype(str).str.strip() == "*").sum()
        for col in numeric_clean_cols
    ],
    "Original_Plus_Count": [
        (df[col].astype(str).str.strip() == "+").sum()
        for col in numeric_clean_cols
    ]
})

cleaning_validation["Expected_Missing"] = (
    cleaning_validation["Original_Star_Count"]
    + cleaning_validation["Original_Plus_Count"]
)

cleaning_validation["Unexpected_Missing"] = (
    cleaning_validation["Missing_After_Cleaning"]
    - cleaning_validation["Expected_Missing"]
)

cleaning_validation[
    cleaning_validation["Unexpected_Missing"] != 0
]

,Column,Missing_After_Cleaning,Original_Star_Count,Original_Plus_Count,Expected_Missing,Unexpected_Missing


In [93]:
# circling back to the (+) values 

master_audit.loc[
    master_audit["Column"] == "BENE_DUAL_PCT",
    "Reason"
] = (
    "Retained as a potentially useful beneficiary characteristic. "
    "Contains 168 CMS '*' suppression values and 8 '+' special values; "
    "both were converted to missing (NaN) in the cleaned dataset. "
    "989 numeric provider observations remain available for analysis."
)

master_audit.loc[
    master_audit["Column"] == "BENE_DUAL_PCT",
    ["Column", "Decision", "Suppressed_Count",
     "Special_Plus_Count", "Reason"]
]

,Column,Decision,Suppressed_Count,Special_Plus_Count,Reason
17,BENE_DUAL_PCT,KEEP,168,8,Retained as a potentially useful beneficiary c...


In [94]:
# spot check

slp_max_provider = df_clean.loc[
    df_clean["SLP_CNCRNT_GRP_PCT"].idxmax(),
    [
        "PRVDR_ID",
        "PRVDR_NAME",
        "STATE",
        "TOT_EPSD_STAY_CNT",
        "TOT_SLP_MNTS",
        "INDVDL_SLP_MNTS",
        "CNCRNT_GRP_SLP_MNTS",
        "COTRT_SLP_MNTS",
        "SLP_MNTS_PER_STAY",
        "SLP_INDVDL_PCT",
        "SLP_CNCRNT_GRP_PCT",
        "SLP_COTRT_PCT"
    ]
]

slp_max_provider

PRVDR_ID                                                          050717
PRVDR_NAME             Lac/rancho Los Amigos National Rehabilitation ...
STATE                                                                 CA
TOT_EPSD_STAY_CNT                                                    127
TOT_SLP_MNTS                                                         420
INDVDL_SLP_MNTS                                                        0
CNCRNT_GRP_SLP_MNTS                                                  420
COTRT_SLP_MNTS                                                         0
SLP_MNTS_PER_STAY                                               3.307087
SLP_INDVDL_PCT                                                       0.0
SLP_CNCRNT_GRP_PCT                                                 100.0
SLP_COTRT_PCT                                                        0.0
Name: 179, dtype: object

In [95]:
pt_max_provider = df_clean.loc[
    df_clean["PT_COTRT_PCT"].idxmax(),
    [
        "PRVDR_ID",
        "PRVDR_NAME",
        "STATE",
        "TOT_EPSD_STAY_CNT",
        "TOT_PT_MNTS",
        "INDVDL_PT_MNTS",
        "CNCRNT_GRP_PT_MNTS",
        "COTRT_PT_MNTS",
        "PT_MNTS_PER_STAY",
        "PT_INDVDL_PCT",
        "PT_CNCRNT_GRP_PCT",
        "PT_COTRT_PCT"
    ]
]

pt_max_provider

PRVDR_ID                                 230031
PRVDR_NAME            Lake Huron Medical Center
STATE                                        MI
TOT_EPSD_STAY_CNT                            73
TOT_PT_MNTS                               36225
INDVDL_PT_MNTS                            11625
CNCRNT_GRP_PT_MNTS                          120
COTRT_PT_MNTS                             24480
PT_MNTS_PER_STAY                     496.232877
PT_INDVDL_PCT                         32.091097
PT_CNCRNT_GRP_PCT                      0.331263
PT_COTRT_PCT                           67.57764
Name: 567, dtype: object

### Manual Review of Extreme Therapy-Delivery Percentages

Two extreme therapy delivery percentages were manually reviewed against the
underlying CMS therapy minute variables to confirm that the engineered features
were calculated correctly.

The provider with 100% concurrent/group SLP reported 420 total SLP minutes,
all of which were concurrent/group minutes. However, overall SLP utilization
was low at approximately 3.31 minutes per stay. The calculated percentage was
therefore retained but should be interpreted alongside overall therapy volume.

The provider with the highest PT co-treatment percentage reported 36,225 total
PT minutes, including 24,480 co-treatment minutes, resulting in approximately
67.58% of PT being delivered as co-treatment. The underlying therapy components
sum correctly to the reported total, supporting the validity of the engineered
percentage.

These extreme values were retained as legitimate reported provider patterns.
They are not interpreted as evidence of inappropriate care or fraud and will
require peer comparison and additional context during later anomaly analysis.

In [96]:
# Final validation of cleaned dataset

print("Final shape:", df_clean.shape)

print(
    "Engineered features present:",
    all(col in df_clean.columns for col in engineered_features)
)

print(
    "Remaining '*' values:",
    (df_clean.astype(str) == "*").sum().sum()
)

print(
    "Remaining '+' values:",
    (df_clean.astype(str) == "+").sum().sum()
)

print(
    "Total missing values:",
    df_clean.isna().sum().sum()
)

Final shape: (1165, 103)
Engineered features present: True
Remaining '*' values: 72
Remaining '+' values: 0
Total missing values: 15312


In [97]:
# confirm work 

remaining_stars = {}

for col in df_clean.columns:
    count = (df_clean[col].astype(str).str.strip() == "*").sum()
    if count > 0:
        remaining_stars[col] = count

pd.DataFrame(
    remaining_stars.items(),
    columns=["Column", "Remaining_Star_Count"]
)

,Column,Remaining_Star_Count
0,TOT_PT_MNTS,6
1,INDVDL_PT_MNTS,6
2,CNCRNT_GRP_PT_MNTS,6
3,COTRT_PT_MNTS,6
4,TOT_OT_MNTS,6
5,INDVDL_OT_MNTS,6
6,CNCRNT_GRP_OT_MNTS,6
7,COTRT_OT_MNTS,6
8,TOT_SLP_MNTS,6
9,INDVDL_SLP_MNTS,6


In [98]:
# Convert raw therapy variables to numeric in the cleaned dataset

for col in therapy_cols:
    df_clean[col] = pd.to_numeric(
        df_clean[col].replace("*", np.nan),
        errors="coerce"
    )

# Confirm no CMS special-value strings remain
print(
    "Remaining '*' values:",
    (df_clean.astype(str) == "*").sum().sum()
)

print(
    "Remaining '+' values:",
    (df_clean.astype(str) == "+").sum().sum()
)

print(
    "Final shape:",
    df_clean.shape
)

Remaining '*' values: 0
Remaining '+' values: 0
Final shape: (1165, 103)


In [99]:
# export both files -- Save cleaned + engineered provider dataset
df_clean.to_csv(
    "CMS_IRF_2024_Cleaned_Engineered.csv",
    index=False
)

# Save variable audit / data dictionary
master_audit.to_csv(
    "CMS_IRF_2024_Variable_Audit.csv",
    index=False
)

print("Files saved successfully.")

Files saved successfully.


In [100]:
df_clean.to_csv(
    "CMS_IRF_2024_Cleaned_Engineered.csv",
    index=False
)

master_audit.to_csv(
    "CMS_IRF_2024_Variable_Audit.csv",
    index=False
)

print("Final Phase 2 files saved successfully.")

Final Phase 2 files saved successfully.
